### Importing of libraries

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import sqlite3
%matplotlib inline
import math
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1_l2
import warnings
warnings.filterwarnings('ignore')

### Data Import

In [2]:
db_path = r"E:\Machine Learning Research\DataModelling\IDA_Data.db"  # <-- change this to your actual .db file path
table_name = "IDA_Data"                    # <-- your table name

conn = sqlite3.connect(db_path)
df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
conn.close()

# === Display===
# print(df.shape)
# print(list(df.columns))
# === Optionally, show first few rows ===
# print(InputHeads)
# print(OutputHeads)
# print("\nFirst 5 rows:")
# print(df.columns)
# df.columns

AllColumns = ['id', 'Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
              'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 'Layer1_FrictionA', 'Layer1_G_kPa', 
              'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion', 'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion',
              'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 
              'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
              'Fundamental_Period', 'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3', 'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6', 'Drift-Y_Level-1', 'Drift-Y_Level-2',
              'Drift-Y_Level-3', 'Drift-Y_Level-4', 'Drift-Y_Level-5', 'Drift-Y_Level-6', 'Displacement-X_Level-1', 'Displacement-X_Level-2', 'Displacement-X_Level-3', 'Displacement-X_Level-4',
              'Displacement-X_Level-5', 'Displacement-X_Level-6', 'Displacement-Y_Level-1', 'Displacement-Y_Level-2', 'Displacement-Y_Level-3', 'Displacement-Y_Level-4', 'Displacement-Y_Level-5',
              'Displacement-Y_Level-6', 'Reaction-Force-X_Level-1', 'Reaction-Force-X_Level-2', 'Reaction-Force-X_Level-3', 'Reaction-Force-X_Level-4', 'Reaction-Force-X_Level-5', 'Reaction-Force-X_Level-6', 
              'Reaction-Force-Y_Level-1', 'Reaction-Force-Y_Level-2', 'Reaction-Force-Y_Level-3', 'Reaction-Force-Y_Level-4', 'Reaction-Force-Y_Level-5', 'Reaction-Force-Y_Level-6', 'Reaction-Moment-X_Level-1',
              'Reaction-Moment-X_Level-2', 'Reaction-Moment-X_Level-3', 'Reaction-Moment-X_Level-4', 'Reaction-Moment-X_Level-5', 'Reaction-Moment-X_Level-6', 'Reaction-Moment-Y_Level-1',
              'Reaction-Moment-Y_Level-2', 'Reaction-Moment-Y_Level-3', 'Reaction-Moment-Y_Level-4', 'Reaction-Moment-Y_Level-5', 'Reaction-Moment-Y_Level-6', 'Rotation-X_Level-1', 'Rotation-X_Level-2',
              'Rotation-X_Level-3', 'Rotation-X_Level-4', 'Rotation-X_Level-5', 'Rotation-X_Level-6', 'Rotation-Y_Level-1', 'Rotation-Y_Level-2', 'Rotation-Y_Level-3', 'Rotation-Y_Level-4', 'Rotation-Y_Level-5',
              'Rotation-Y_Level-6', 'Rotation-Z_Level-1', 'Rotation-Z_Level-2', 'Rotation-Z_Level-3', 'Rotation-Z_Level-4', 'Rotation-Z_Level-5', 'Rotation-Z_Level-6', 'Torsional-Irregularity-Ratio_Level-1',
              'Torsional-Irregularity-Ratio_Level-2', 'Torsional-Irregularity-Ratio_Level-3', 'Torsional-Irregularity-Ratio_Level-4', 'Torsional-Irregularity-Ratio_Level-5', 'Torsional-Irregularity-Ratio_Level-6',
              'Max-Uplift_Level-1', 'Max-Uplift-Point_Level-1', 'Max-Settlement_Level-1', 'Max-Settlement-Point_Level-1', 'Max-Pseudo-Time_Level-1']

### Data Processing

In [3]:
def dataPreparation(DoNormalize = True,Z_ScoreScaler = True    ,        ModelSet = 2     ,includeCategoricalData = False  ):
    
    # Creating of the PeakGroundAcceleration column on the database by product of ScaleFactor and MaximumAcceleration of the data
    df['maxacceleration_mps2'] = pd.to_numeric(df['maxacceleration_mps2'], errors='coerce')
    df['ScaleFactor'] = pd.to_numeric(df['ScaleFactor'], errors='coerce')
    df['PeakGroundAcceleration'] = df['maxacceleration_mps2'] * df['ScaleFactor']
    
    #Creating of the Prediction column as Max drift of the all levels merger in single column as below
    drift_cols = [    'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3',    'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6']
    df[drift_cols] = df[drift_cols].apply(pd.to_numeric, errors='coerce')
    df['Max_Drift_X'] = df[drift_cols].max(axis=1)
    
    # Check for any NaNs that might have appeared due to conversion issues
    # print(df[['maxacceleration_mps2', 'ScaleFactor', 'PeakGroundAcceleration']].sample(50))
    # print("Number of NaNs in new column:", df['PeakGroundAcceleration'].isna().sum())
    
    InputHeadsAvailableAll = ['Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
                  'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                  'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2',
                  'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
                  'Fundamental_Period']
    InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   'StiffnessX_Total', 'Plan-area', 'Seismic-weight', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                          
                  "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                             'Fundamental_Period']
    # InputHeadsSelected = InputHeadsAvailableAll

    
    #Best performer for the fixedbase and displacement output
                   # "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                   #           'Fundamental_Period']
    #Best performer for the maximum drift along X with r2 as 89.36 (Donot include  'Plan-area', 'Seismic-weight', 'StiffnessX_Total', as they donot produce any effect and only add burden to model)
    # InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                          
    #               "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
    #                          'Fundamental_Period']
    #Best performer for the fixed and output as Reaction-Force-X_Level-1:  r2 as 0.95 max with 15 features
    # InputHeadsSelected = ['Earthquake', 'Building', 'Fundamental_Period',  'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx',
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
    #                "PeakGroundAcceleration", 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',
    #                         ]
    #For soil analysis take Rup, Vs30... Use maxacceleration_mps2 that represents the pga value of site and for the building specific analysis ie performance
    # Levels measurement take PSA, PSV and PSD values over the peac acc peak vel and peak disp and so on 
    #Final selected features (9): ['ScaleFactor', 'Vs30', 'cav_gs', 'arias_mps', 'husid_s', 'hous_m', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps']
    
    
    
    #_______________________________________________________________________________________________________________________________________________________________Processing starts from here
    if ModelSet == 1 :
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")] 
    elif ModelSet == 3:
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")]
        InputHeads = InputHeads + ["BaseCondition"]
    else:
        InputHeads = InputHeadsSelected
    # OutputHeads = ['Displacement-X_Level-5']
    # OutputHeads = ["Reaction-Force-X_Level-1"]
    OutputHeads = ['Max_Drift_X']
    
    # df[InputHeads].info
    
    ####__________________________________________________________________________________________________________________________________Data selection 
    
    # Extract all the data from the fixed base conditions only
    if ModelSet == 1:
        df_fixed = df[df['BaseCondition'] == 'Fixed']
    elif ModelSet == 2:
        df_fixed = df[df['BaseCondition'] != 'Fixed']
    else:
        df_fixed = df[df['BaseCondition'].isin(['Fixed', 'Soft', 'Medium', 'Hard'])]
        
    
    #Convert all the data in database to numerical format
    if includeCategoricalData:
        df_numeric = df_fixed.copy()
        for col in df_numeric.columns:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='ignore')
    else:
        df_numeric = df_fixed.apply(pd.to_numeric, errors='coerce')
        df_numeric.shape
    # df_numeric.head
    
    ##Extract the data if the scale factor is less than or equal to 1 only
    if ModelSet != 1:
        df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    
    #Removing all the white spaces on the column titles
    df_numeric.columns = df_numeric.columns.str.replace(" ", "_")

    ####__________________________________________________________________________________________________________________________________Encoding of data
    # # Normalize for the data of the features as seismic weights, and so on
    features_to_normalize = [x for x in InputHeads
                             if pd.api.types.is_numeric_dtype(df_numeric[x])]
    
    
    #One hot encoding of the categorical data if user set it to include in the model
    if includeCategoricalData:
        Catcolumns =  [x for x in InputHeads if x not in features_to_normalize]
        # print(Catcolumns)

        if Catcolumns:
            data_toEncode = df_numeric[Catcolumns]     
            CatDataEncoded = pd.get_dummies(data_toEncode, columns = Catcolumns,  dtype=float)
            df_numeric = df_numeric.drop(columns=Catcolumns)

        
            if df_numeric.shape[0] == CatDataEncoded.shape[0]:
                InputHeads = InputHeads + list(CatDataEncoded.columns)
                df_numeric = pd.concat([df_numeric, CatDataEncoded], axis=1)

            else: 
                raise
    

    
    ## Converts all the data to the numeric even to that of hot encoded as they are in float
    df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')
        
    
    ####__________________________________________________________________________________________________________________________________Missing Data Handling
    df_numeric =df_numeric.dropna(axis=1, how='all')             #Dropping for all the columns (axis = 1) if it has all values of NaN
    
    #### Show only columns that actually contain NaN
    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    # nan_rows = df_numeric[df_numeric.isna().any(axis=1)]
    # print("Rows containing NaN values:")
    # print(nan_rows)
    
    df_numeric = df_numeric.apply(lambda col: col.fillna(col.median()) if col.dtype != 'object' else col)   #For any value other than object if the data is shown as missing then it fills up with the median of that column

    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    df_numeric = df_numeric.dropna()                             #Now also if any of the Nan data persists then the system drops that specific rows
    

    ####__________________________________________________________________________________________________________________________________Normalization of data
    if DoNormalize:
        scaler = MinMaxScaler(feature_range=(0, 1))
        if Z_ScoreScaler:
            scaler = StandardScaler()
        cols_to_scale = [c for c in features_to_normalize if c in df_numeric.columns] # Only normalize columns that actually exist in df
        df_numeric[cols_to_scale] = scaler.fit_transform(df_numeric[cols_to_scale])

    ####__________________________________________________________________________________________________________________________________Assignment of the training data
    
    # # Extract for the data to be in x and y variables
    X = df_numeric[[col for col in InputHeads if col in df_numeric.columns]]
    y = df_numeric[[col for col in OutputHeads if col in df_numeric.columns]]

    df_numeric = pd.concat([X, y], axis=1)  # combine X and y

    
    # X.shape
    # print((X.columns))
    # cols = CatDataEncoded.columns
    # for co in cols:
    #     print(co)
    # print(CatDataEncoded.columns)
    # df_numeric.sample(10)
    # X.shape

    return X, y



### Getting data Statistics

In [4]:
import pandas as pd
import numpy as np

def generate_complete_data_statistics(X, y):
    """Generate comprehensive statistical summary for entire dataset"""
    
    # Convert to proper formats
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("COMPLETE DATASET STATISTICAL SUMMARY")
    print("="*80)
    print(f"{'Dataset':<20} {'Samples':<10} {'Features':<10}")
    print(f"{'':<20} {X_clean.shape[0]:<10} {X_clean.shape[1]:<10}")
    print("="*80)
    
    # Features summary
    print(f"\n{'FEATURES SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    for column in X_clean.columns:
        count = len(X_clean[column])
        mean = X_clean[column].mean()
        std = X_clean[column].std()
        min_val = X_clean[column].min()
        max_val = X_clean[column].max()
        
        print(f"{column:<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Target variable summary
    print(f"\n{'TARGET VARIABLE SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    count = len(y_clean)
    mean = y_clean.mean()
    std = y_clean.std()
    min_val = y_clean.min()
    max_val = y_clean.max()
    
    print(f"{'Target':<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Additional statistics
    print(f"\n{'ADDITIONAL STATISTICS':<50}")
    print("-"*50)
    print(f"{'Metric':<25} {'Value':<25}")
    print("-"*50)
    print(f"{'Total Samples':<25} {X_clean.shape[0]:<25}")
    print(f"{'Total Features':<25} {X_clean.shape[1]:<25}")
    print(f"{'Missing Values in X':<25} {X_clean.isnull().sum().sum():<25}")
    print(f"{'Missing Values in y':<25} {np.isnan(y_clean).sum():<25}")
    print(f"{'Target Skewness':<25} {pd.Series(y_clean).skew():<25.4f}")
    print(f"{'Target Kurtosis':<25} {pd.Series(y_clean).kurtosis():<25.4f}")

# Quick version for compact output
def quick_data_stats(X, y):
    """Quick statistical summary"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("DATASET STATISTICS")
    print("="*90)
    print(f"{'Parameter':<20} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10} {'Type':<10}")
    print("-"*90)
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column:<20} {len(data):<8} {data.mean():<12.4f} {data.std():<12.4f} "
              f"{data.min():<10.2f} {data.max():<10.2f} {'Feature':<10}")
    
    # Target
    print(f"{'Target':<20} {len(y_clean):<8} {y_clean.mean():<12.4f} {y_clean.std():<12.4f} "
          f"{y_clean.min():<10.2f} {y_clean.max():<10.2f} {'Target':<10}")

# Generate LaTeX table for research paper
def generate_latex_table(X, y):
    """Generate LaTeX table for research paper"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("\nLATEX TABLE FOR RESEARCH PAPER:")
    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\caption{Descriptive Statistics of the Complete Dataset}")
    print("\\label{tab:data_statistics}")
    print("\\begin{tabular}{lrrrrr}")
    print("\\hline")
    print("Parameter & Count & Mean & Std & Min & Max \\\\")
    print("\\hline")
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column} & {len(data)} & {data.mean():.4f} & {data.std():.4f} & {data.min():.2f} & {data.max():.2f} \\\\")
    
    # Target
    print(f"Target & {len(y_clean)} & {y_clean.mean():.4f} & {y_clean.std():.4f} & {y_clean.min():.2f} & {y_clean.max():.2f} \\\\")
    print("\\hline")
    print("\\end{tabular}")
    print("\\end{table}")



# Singular Plots

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.offsetbox import AnchoredText
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    mean_absolute_percentage_error, explained_variance_score
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import learning_curve, cross_val_score, validation_curve
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# Optional imports
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


class NeuralNetworkModernVisualizer:
    """
    Complete visualization suite for Neural Network (Keras/TensorFlow) models
    with modern color scheme – identical structure and styling as LightGBM visualizer.
    """

    def __init__(self, model, X_train, X_test, y_train, y_test, feature_names=None,
                 history=None, results_path="./neural_network_modern_plots"):
        """
        Parameters
        ----------
        model : keras.Model
            Trained neural network model.
        X_train, X_test : array-like
            Training and test features.
        y_train, y_test : array-like
            Training and test targets.
        feature_names : list, optional
            Names of features. If None, inferred from X_train if DataFrame.
        history : keras.callbacks.History, optional
            Training history containing loss, val_loss, lr etc.
        results_path : str
            Directory to save plots.
        """
        # Convert to numpy arrays
        self.X_train = np.array(X_train)
        self.X_test = np.array(X_test)
        self.y_train = np.array(y_train).ravel()
        self.y_test = np.array(y_test).ravel()
        self.model = model
        self.history = history

        # Feature names
        if feature_names is None:
            if hasattr(X_train, 'columns'):
                self.feature_names = list(X_train.columns)
            else:
                self.feature_names = [f'Feature_{i}' for i in range(self.X_train.shape[1])]
        else:
            self.feature_names = list(feature_names)

        self.results_path = results_path
        os.makedirs(results_path, exist_ok=True)

        # Predictions and residuals
        self.y_pred_train = self.model.predict(self.X_train, verbose=0).flatten()
        self.y_pred_test = self.model.predict(self.X_test, verbose=0).flatten()
        self.residuals_train = self.y_train - self.y_pred_train
        self.residuals_test = self.y_test - self.y_pred_test

        # Metrics
        self.metrics = {
            'train': {
                'R2': r2_score(self.y_train, self.y_pred_train),
                'RMSE': np.sqrt(mean_squared_error(self.y_train, self.y_pred_train)),
                'MAE': mean_absolute_error(self.y_train, self.y_pred_train),
                'MAPE': mean_absolute_percentage_error(self.y_train, self.y_pred_train),
                'Explained Variance': explained_variance_score(self.y_train, self.y_pred_train)
            },
            'test': {
                'R2': r2_score(self.y_test, self.y_pred_test),
                'RMSE': np.sqrt(mean_squared_error(self.y_test, self.y_pred_test)),
                'MAE': mean_absolute_error(self.y_test, self.y_pred_test),
                'MAPE': mean_absolute_percentage_error(self.y_test, self.y_pred_test),
                'Explained Variance': explained_variance_score(self.y_test, self.y_pred_test)
            }
        }

        # Modern vibrant color palette (identical to LightGBM)
        self.colors = {
            'primary': '#FF00FF',      # magenta
            'secondary': '#7FFF00',    # chartreuse
            'accent': '#32CD32',       # limegreen
            'danger': '#FFBF00',       # amber
            'warning': '#9370DB',      # medium purple
            'info': '#00FFFF',         # cyan
            'dark': '#1F51FF',         # blue
            'light': '#FFFFFF',        # white
            'success': '#4B0082',      # indigo
            'purple': '#FF073A',       # emerald green
        }

        # Gradients (same)
        self.gradients = {
            'train': ['#FF00FF', '#CC00CC'],
            'test': ['#32CD32', '#CC0630'],
            'feature': ['#FFBF00', '#28A428'],
            'residual': ['#00FFFF', '#7659B0']
        }

        self.set_modern_style()

    # ==================== STYLING METHODS (identical to LightGBM) ====================

    def set_modern_style(self):
        """Set modern, clean plot style identical to LightGBM visualizer"""
        plt.style.use('default')
        mpl.rcParams.update({
            'font.family': 'sans-serif',
            'font.sans-serif': ['Times New Roman', 'DejaVu Sans', 'Helvetica'],
            'font.size': 11,
            'figure.dpi': 300,
            'figure.figsize': (8, 6),
            'figure.titlesize': 16,
            'figure.titleweight': 'bold',

            # Axes
            'axes.labelsize': 13,
            'axes.titlesize': 14,
            'axes.titleweight': 'bold',
            'axes.linewidth': 1.2,
            'axes.grid': True,
            'grid.alpha': 0.15,
            'grid.linestyle': '-',
            'grid.linewidth': 0.8,

            # Ticks
            'xtick.labelsize': 11,
            'ytick.labelsize': 11,
            'xtick.direction': 'out',
            'ytick.direction': 'out',
            'xtick.major.size': 5,
            'ytick.major.size': 5,
            'xtick.major.width': 1,
            'ytick.major.width': 1,

            # Legend
            'legend.fontsize': 11,
            'legend.frameon': True,
            'legend.framealpha': 0.95,
            'legend.edgecolor': '#E5E7EB',
            'legend.fancybox': False,
            'legend.facecolor': 'white',

            # Lines
            'lines.linewidth': 2.5,
            'lines.markersize': 6,
            'lines.markeredgewidth': 1,
            'lines.markeredgecolor': 'white',
        })

    def create_modern_textbox(self, ax, text, location='upper left', fontsize=10,
                              bg_color='white', alpha=0.95, border=False):
        """Modern anchored text box"""
        if border:
            box_style = dict(boxstyle="round,pad=0.4", facecolor=bg_color,
                             alpha=alpha, edgecolor='#E5E7EB', linewidth=1.5)
        else:
            box_style = dict(boxstyle="round,pad=0.4", facecolor=bg_color,
                             alpha=alpha, edgecolor='none')
        text_box = AnchoredText(text, loc=location, frameon=border,
                                pad=0.6, borderpad=0.8,
                                prop=dict(fontsize=fontsize, bbox=box_style))
        ax.add_artist(text_box)
        return text_box

    def save_modern_plot(self, fig, name, subfolder='', dpi=300):
        """Save plot with modern formatting"""
        save_dir = os.path.join(self.results_path, subfolder) if subfolder else self.results_path
        os.makedirs(save_dir, exist_ok=True)
        filepath = os.path.join(save_dir, f"{name}.png")
        fig.tight_layout(pad=2.0)
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight',
                    facecolor='white', edgecolor='none', pad_inches=0.2)
        plt.close(fig)
        print(f"  ✓ Saved: {name}.png")
        return fig

    def _remove_spines(self, ax):
        """Remove top and right spines for cleaner look"""
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1)
        ax.spines['bottom'].set_linewidth(1)

    def get_feature_name(self, idx):
        """Clean feature name for display"""
        if isinstance(idx, int) and idx < len(self.feature_names):
            name = self.feature_names[idx]
        elif isinstance(idx, str):
            name = idx
        else:
            return f'Feature_{idx}'
        name = str(name).replace('_', ' ').title()
        name = name.split('.')[0]
        if len(name) > 25:
            name = name[:22] + '...'
        return name

    # ==================== 1. EDA PLOTS (identical to LightGBM) ====================

    def plot_correlation_heatmap(self, figsize=(12, 10)):
        """Feature correlation heatmap (green‑white‑red diverging)"""
        df = pd.DataFrame(self.X_train, columns=self.feature_names)
        df['Target'] = self.y_train
        corr = df.corr()

        from matplotlib.colors import LinearSegmentedColormap
        colors = [(0, 1, 0), (1, 1, 1), (1, 0, 0)]
        custom_cmap = LinearSegmentedColormap.from_list('custom_diverging', colors, N=256)

        fig, ax = plt.subplots(figsize=figsize)
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, cmap=custom_cmap, center=0,
                    square=True, linewidths=0.3, cbar_kws={"shrink": 0.8, "label": "Correlation"},
                    annot=True, fmt='.2f', ax=ax,
                    annot_kws={'size': 7, 'weight': 'bold' if corr.shape[0] < 15 else 'normal'},
                    vmin=-1, vmax=1)
        ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14, pad=15)
        ax.set_xlabel('Features', fontsize=12)
        ax.set_ylabel('Features', fontsize=12)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
        plt.setp(ax.yaxis.get_majorticklabels(), rotation=0, fontsize=9)
        ax.tick_params(length=0)

        cbar = ax.collections[0].colorbar
        cbar.set_label('Correlation', fontsize=11)
        cbar.ax.tick_params(labelsize=9)

        self.save_modern_plot(fig, '01_correlation_heatmap', '')
        return fig

    def plot_distributions(self, figsize=(15, 12)):
        """Distribution of target and top features"""
        n_features = min(self.X_train.shape[1], 12)
        n_cols = 3
        n_rows = (n_features + n_cols) // n_cols + 1
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = axes.flatten()

        # Target
        ax = axes[0]
        sns.histplot(self.y_train, kde=True, color=self.colors['primary'], ax=ax, bins=30)
        ax.axvline(np.mean(self.y_train), color='red', linestyle='--',
                   label=f'Mean: {np.mean(self.y_train):.2f}')
        ax.axvline(np.median(self.y_train), color='green', linestyle='--',
                   label=f'Median: {np.median(self.y_train):.2f}')
        ax.set_title('Target Distribution', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        # Features
        for i in range(n_features):
            ax = axes[i+1]
            data = self.X_train[:, i]
            sns.histplot(data, kde=True, color=self.colors['secondary'], ax=ax, bins=30)
            ax.set_title(self.get_feature_name(i), fontweight='bold')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)

        for i in range(n_features+1, len(axes)):
            axes[i].set_visible(False)

        fig.suptitle('Feature and Target Distributions', fontweight='bold', y=1.02)
        self.save_modern_plot(fig, '02_distributions', '')
        return fig

    def plot_boxplots(self, max_categories=4, figsize=(14, 8)):
        """Box plots by top‑variance features"""
        variances = np.var(self.X_train, axis=0)
        top_indices = np.argsort(variances)[-max_categories:]
        fig, axes = plt.subplots(1, max_categories, figsize=figsize)
        if max_categories == 1:
            axes = [axes]

        df = pd.DataFrame(self.X_train, columns=self.feature_names)
        df['target'] = self.y_train

        for i, idx in enumerate(top_indices):
            ax = axes[i]
            col = self.feature_names[idx]
            if df[col].nunique() > 10:
                df[f'{col}_bin'] = pd.qcut(df[col], q=5, duplicates='drop')
                plot_col = f'{col}_bin'
            else:
                plot_col = col
            sns.boxplot(x=plot_col, y='target', data=df, ax=ax, palette='Set2')
            ax.set_title(f'Target by {self.get_feature_name(col)}', fontweight='bold')
            ax.set_xlabel(self.get_feature_name(col))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)

        fig.suptitle('Target Distribution by Categories', fontweight='bold', y=1.02)
        self.save_modern_plot(fig, '03_boxplots', '')
        return fig

    def plot_feature_scatter(self, top_features=6, figsize=(15, 10)):
        """Scatter plots of most correlated features vs target"""
        correlations = []
        for i in range(self.X_train.shape[1]):
            corr = np.corrcoef(self.X_train[:, i], self.y_train)[0, 1]
            correlations.append(abs(corr) if not np.isnan(corr) else 0)
        top_indices = np.argsort(correlations)[-top_features:][::-1]

        n_cols = 3
        n_rows = (len(top_indices) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = axes.flatten()

        for i, idx in enumerate(top_indices):
            ax = axes[i]
            ax.scatter(self.X_train[:, idx], self.y_train, alpha=0.5, s=20,
                       c=self.colors['primary'], edgecolors='white', linewidth=0.5)
            z = np.polyfit(self.X_train[:, idx], self.y_train, 1)
            p = np.poly1d(z)
            x_range = np.linspace(self.X_train[:, idx].min(), self.X_train[:, idx].max(), 100)
            ax.plot(x_range, p(x_range), 'r--', alpha=0.8,
                    label=f'Corr: {correlations[idx]:.3f}')
            ax.set_xlabel(self.get_feature_name(idx))
            ax.set_ylabel('Target')
            ax.set_title(f'{self.get_feature_name(idx)} vs Target', fontweight='bold')
            ax.legend(fontsize=8, framealpha=0.95, edgecolor='#E5E7EB')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)

        for i in range(len(top_indices), len(axes)):
            axes[i].set_visible(False)

        fig.suptitle('Feature vs Target Relationships', fontweight='bold', y=1.02)
        self.save_modern_plot(fig, '04_feature_scatter', '')
        return fig

    def plot_missing_values(self, figsize=(12, 6)):
        """Missing value bar chart"""
        fig, ax = plt.subplots(figsize=figsize)
        nan_count = np.isnan(self.X_train).sum(axis=0)
        if nan_count.sum() > 0:
            ax.barh(self.feature_names, nan_count, color=self.colors['danger'])
            ax.set_xlabel('Number of Missing Values')
            ax.set_title('Missing Values by Feature', fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No Missing Values Detected', ha='center', va='center',
                    transform=ax.transAxes, fontsize=14)
            ax.set_title('Missing Values Analysis', fontweight='bold')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_modern_plot(fig, '05_missing_values', '')
        return fig

    def plot_outlier_detection(self, figsize=(14, 6)):
        """IQR outlier detection for target variable"""
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        Q1 = np.percentile(self.y_train, 25)
        Q3 = np.percentile(self.y_train, 75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = (self.y_train < lower) | (self.y_train > upper)

        ax = axes[0]
        bp = ax.boxplot(self.y_train, patch_artist=True, widths=0.6)
        bp['boxes'][0].set_facecolor(self.colors['light'])
        bp['boxes'][0].set_edgecolor(self.colors['primary'])
        bp['medians'][0].set_color(self.colors['danger'])
        ax.set_title('Box Plot (IQR Outliers)', fontweight='bold')
        ax.set_ylabel('Target')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        ax = axes[1]
        colors = np.where(outliers, self.colors['danger'], self.colors['primary'])
        ax.scatter(range(len(self.y_train)), self.y_train, c=colors, alpha=0.6, s=30,
                   edgecolors='white', linewidth=0.5)
        ax.axhline(y=upper, color='red', linestyle='--', label='Upper bound')
        ax.axhline(y=lower, color='red', linestyle='--', label='Lower bound')
        ax.set_xlabel('Sample Index')
        ax.set_ylabel('Target')
        ax.set_title(f'Outliers: {outliers.sum()} ({outliers.sum()/len(self.y_train)*100:.1f}%)', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        fig.suptitle('Outlier Detection Analysis', fontweight='bold')
        self.save_modern_plot(fig, '06_outlier_detection', '')
        return fig

    # ==================== 2. PERFORMANCE PLOTS (identical to LightGBM) ====================

    def plot_actual_vs_predicted(self, figsize=(10, 8)):
        """Actual vs predicted scatter with perfect line, combined regression, and confidence band"""
        fig, ax = plt.subplots(figsize=figsize)

        ax.scatter(self.y_train, self.y_pred_train, alpha=0.6, s=40,
                   c=self.gradients['train'][0], edgecolors='white', linewidth=0.8,
                   label='Train')
        ax.scatter(self.y_test, self.y_pred_test, alpha=0.7, s=50,
                   c=self.gradients['test'][0], edgecolors='white', linewidth=0.8,
                   label='Test')

        min_val = min(self.y_train.min(), self.y_test.min())
        max_val = max(self.y_train.max(), self.y_test.max())
        margin = 0.05 * (max_val - min_val)
        ax.plot([min_val - margin, max_val + margin],
                [min_val - margin, max_val + margin],
                'k--', alpha=0.5, linewidth=1.5, label='Perfect (y=x)')

        # Combined regression
        all_actual = np.concatenate([self.y_train, self.y_test])
        all_pred = np.concatenate([self.y_pred_train, self.y_pred_test])
        z = np.polyfit(all_actual, all_pred, 1)
        p = np.poly1d(z)
        x_line = np.linspace(min_val - margin, max_val + margin, 100)
        ax.plot(x_line, p(x_line), '-', color=self.colors['dark'], linewidth=2.5, alpha=0.9,
                label=f'Combined fit: y={z[0]:.3f}x+{z[1]:.3f}')

        # Confidence band (based on test residuals)
        resid_std = np.std(self.residuals_test)
        ax.fill_between(x_line, p(x_line) - 2*resid_std, p(x_line) + 2*resid_std,
                        alpha=0.1, color=self.colors['secondary'], label='±2σ band')

        ax.set_xlabel('Actual', fontweight='bold', fontsize=12)
        ax.set_ylabel('Predicted', fontweight='bold', fontsize=12)
        ax.set_title('Actual vs Predicted Values', fontweight='bold', fontsize=14, pad=15)
        ax.legend(loc='lower right', framealpha=0.95, edgecolor='#E5E7EB', fontsize=9)
        ax.grid(True, alpha=0.1)

        text = (f"Train: R²={self.metrics['train']['R2']:.4f}  RMSE={self.metrics['train']['RMSE']:.4f}\n"
                f"Test:  R²={self.metrics['test']['R2']:.4f}  RMSE={self.metrics['test']['RMSE']:.4f}")
        self.create_modern_textbox(ax, text, 'upper left', fontsize=10, bg_color='white', border=False)
        eq_text = f"Combined regression: y = {z[0]:.4f}x + {z[1]:.4f}"
        self.create_modern_textbox(ax, eq_text, 'lower center', fontsize=10, bg_color='white', border=True)

        self._remove_spines(ax)
        self.save_modern_plot(fig, '07_actual_vs_predicted', '')
        return fig

    def plot_residuals_vs_predicted(self, figsize=(10, 8)):
        """Residuals vs predicted values"""
        fig, ax = plt.subplots(figsize=figsize)
        ax.scatter(self.y_pred_train, self.residuals_train, alpha=0.5, s=30,
                   c=self.colors['primary'], edgecolors='white', label='Train')
        ax.scatter(self.y_pred_test, self.residuals_test, alpha=0.5, s=30,
                   c=self.colors['secondary'], edgecolors='white', label='Test')
        ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
        ax.set_xlabel('Predicted', fontweight='bold')
        ax.set_ylabel('Residuals', fontweight='bold')
        ax.set_title('Residuals vs Predicted', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_modern_plot(fig, '08_residuals_vs_predicted', '')
        return fig

    def plot_residual_distribution(self, figsize=(12, 5)):
        """Residual distribution histogram + Q-Q plot"""
        fig, axes = plt.subplots(1, 2, figsize=figsize)

        ax = axes[0]
        sns.histplot(self.residuals_train, kde=True, color=self.colors['primary'],
                     label='Train', alpha=0.5, bins=30, ax=ax)
        sns.histplot(self.residuals_test, kde=True, color=self.colors['secondary'],
                     label='Test', alpha=0.5, bins=30, ax=ax)
        ax.axvline(0, color='red', linestyle='--')
        ax.set_xlabel('Residuals')
        ax.set_title('Residual Distribution', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        ax = axes[1]
        stats.probplot(self.residuals_test, dist="norm", plot=ax)
        ax.get_lines()[0].set_marker('o')
        ax.get_lines()[0].set_markersize(4)
        ax.get_lines()[0].set_markerfacecolor(self.colors['secondary'])
        ax.get_lines()[1].set_color('red')
        ax.get_lines()[1].set_linewidth(2)
        ax.set_title('Q-Q Plot (Test)', fontweight='bold')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        fig.suptitle('Residual Analysis - Normality Check', fontweight='bold')
        self.save_modern_plot(fig, '09_residual_distribution', '')
        return fig

    def plot_learning_curves(self, cv=5, figsize=(10, 6)):
        """
        Learning curves using sklearn's learning_curve.
        Note: This will refit the model many times (slow for NN). Consider using history instead.
        For NN, you may set cv=1 or use the training history plot.
        """
        try:
            train_sizes, train_scores, test_scores = learning_curve(
                self.model, self.X_train, self.y_train, cv=min(cv, 3), n_jobs=-1,
                train_sizes=np.linspace(0.1, 1.0, 10),
                scoring='neg_mean_squared_error'
            )
            fig, ax = plt.subplots(figsize=figsize)
            train_scores_mean = -np.mean(train_scores, axis=1)
            test_scores_mean = -np.mean(test_scores, axis=1)
            train_std = np.std(train_scores, axis=1)
            test_std = np.std(test_scores, axis=1)

            ax.fill_between(train_sizes, train_scores_mean - train_std,
                            train_scores_mean + train_std, alpha=0.2, color=self.colors['primary'])
            ax.fill_between(train_sizes, test_scores_mean - test_std,
                            test_scores_mean + test_std, alpha=0.2, color=self.colors['secondary'])
            ax.plot(train_sizes, train_scores_mean, 'o-', color=self.colors['primary'], label='Train')
            ax.plot(train_sizes, test_scores_mean, 'o-', color=self.colors['secondary'], label='Validation')
            ax.set_xlabel('Training Examples')
            ax.set_ylabel('Mean Squared Error')
            ax.set_title('Learning Curves', fontweight='bold')
            ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
            self.save_modern_plot(fig, '10_learning_curves', '')
            return fig
        except Exception as e:
            print(f"  Learning curves skipped (NN may need more custom implementation): {e}")
            return None

    def plot_cumulative_error(self, figsize=(10, 6)):
        """Cumulative error distribution"""
        fig, ax = plt.subplots(figsize=figsize)
        errors = np.abs(self.residuals_test)
        sorted_errors = np.sort(errors)
        cumulative = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors) * 100
        ax.plot(sorted_errors, cumulative, 'o-', color=self.colors['primary'], linewidth=2, markersize=4)
        ax.set_xlabel('Absolute Error')
        ax.set_ylabel('Cumulative Percentage (%)')
        ax.set_title('Cumulative Error Distribution', fontweight='bold')
        ax.grid(True, alpha=0.1)

        mae = self.metrics['test']['MAE']
        pct_mae = np.sum(errors <= mae) / len(errors) * 100
        ax.axvline(x=mae, color='red', linestyle='--', alpha=0.7)
        ax.text(mae + 0.01, 50, f'MAE: {mae:.4f}\n{pct_mae:.1f}% within', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
        self._remove_spines(ax)
        self.save_modern_plot(fig, '11_cumulative_error', '')
        return fig

    # ==================== 3. FEATURE IMPORTANCE (same as LightGBM) ====================

    def plot_top_features(self, top_n=15, figsize=(6, 8)):
        """
        Top feature importance using permutation importance.
        For neural networks, built-in feature importance is not available,
        so we use permutation importance.
        """
        try:
            perm_importance = permutation_importance(
                self.model, self.X_test, self.y_test,
                n_repeats=10, random_state=42, scoring='r2', n_jobs=-1
            )
            feature_importance = perm_importance.importances_mean
            sorted_idx = np.argsort(feature_importance)[::-1][:top_n]
            sorted_importance = feature_importance[sorted_idx]
            sorted_names = [self.feature_names[i] for i in sorted_idx]

            fig, ax = plt.subplots(figsize=figsize)
            y_pos = np.arange(len(sorted_idx))
            colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_idx)))
            ax.barh(y_pos, sorted_importance, color=colors, alpha=0.8, edgecolor='white')
            ax.set_yticks(y_pos)
            ax.set_yticklabels([self.get_feature_name(n) for n in sorted_names])
            ax.invert_yaxis()
            ax.set_xlabel('Permutation Importance (Δ R²)', fontweight='bold')
            ax.set_title('Top Feature Importances', fontweight='bold')
            ax.grid(True, alpha=0.1, axis='x')

            for i, val in enumerate(sorted_importance):
                ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
            self._remove_spines(ax)
            self.save_modern_plot(fig, '12_top_features', '')
            return fig
        except Exception as e:
            print(f"  Top features failed: {e}")
            return None

    def plot_cumulative_features(self, figsize=(6, 6)):
        """Cumulative feature importance from permutation importance"""
        try:
            perm_importance = permutation_importance(
                self.model, self.X_test, self.y_test,
                n_repeats=10, random_state=42, scoring='r2', n_jobs=-1
            )
            feature_importance = perm_importance.importances_mean
            sorted_importance = np.sort(feature_importance)[::-1]
            cumulative = np.cumsum(sorted_importance) / np.sum(feature_importance)

            fig, ax = plt.subplots(figsize=figsize)
            ax.plot(range(1, len(cumulative) + 1), cumulative, 'o-',
                    color=self.colors['primary'], linewidth=2, markersize=4)
            ax.set_xlabel('Number of Features', fontweight='bold')
            ax.set_ylabel('Cumulative Importance', fontweight='bold')
            ax.set_title('Cumulative Feature Importance', fontweight='bold')
            ax.grid(True, alpha=0.1)

            for threshold in [0.8, 0.9, 0.95]:
                idx = np.where(cumulative >= threshold)[0]
                if len(idx) > 0:
                    n_features = idx[0] + 1
                    ax.axvline(x=n_features, color='red', linestyle='--', alpha=0.5)
                    ax.text(n_features, threshold, f'{n_features}', ha='center', va='bottom')
            self._remove_spines(ax)
            self.save_modern_plot(fig, '13_cumulative_features', '')
            return fig
        except Exception as e:
            print(f"  Cumulative features failed: {e}")
            return None

    def plot_permutation_importance(self, n_repeats=10, top_n=20, figsize=(12, 8)):
        """Permutation importance with error bars (same as LightGBM)"""
        try:
            perm_importance = permutation_importance(
                self.model, self.X_test, self.y_test,
                n_repeats=n_repeats, random_state=42, n_jobs=-1
            )
            sorted_idx = perm_importance.importances_mean.argsort()[::-1][:top_n]
            fig, ax = plt.subplots(figsize=figsize)
            y_pos = np.arange(len(sorted_idx))
            means = perm_importance.importances_mean[sorted_idx]
            stds = perm_importance.importances_std[sorted_idx]
            feature_labels = [self.get_feature_name(self.feature_names[i]) for i in sorted_idx]

            ax.barh(y_pos, means, xerr=stds, capsize=3,
                    color=plt.cm.plasma(np.linspace(0.2, 0.8, len(sorted_idx))),
                    edgecolor='white', linewidth=1)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(feature_labels)
            ax.invert_yaxis()
            ax.set_xlabel('Permutation Importance (Δ R²)', fontweight='bold')
            ax.set_title('Permutation Feature Importance', fontweight='bold')
            ax.grid(True, alpha=0.1, axis='x')
            self._remove_spines(ax)
            self.save_modern_plot(fig, '14_permutation_importance', '')
            return fig
        except Exception as e:
            print(f"  Permutation importance failed: {e}")
            return None

    def plot_shap_summary(self, sample_size=200, top_features=15, figsize=(12, 8)):
        """SHAP summary plot (if available)"""
        if not SHAP_AVAILABLE:
            print("  SHAP not installed. Skipping SHAP plot.")
            return None
        try:
            n_samples = min(sample_size, len(self.X_test))
            indices = np.random.choice(len(self.X_test), n_samples, replace=False)
            X_sample = self.X_test[indices]
            # Use a wrapper for keras model prediction
            def predict_fn(x):
                return self.model.predict(x, verbose=0).flatten()
            explainer = shap.Explainer(predict_fn, X_sample)
            shap_values = explainer(X_sample)
            fig, ax = plt.subplots(figsize=figsize)
            shap.summary_plot(shap_values, X_sample,
                              feature_names=[self.get_feature_name(i) for i in range(X_sample.shape[1])],
                              max_display=top_features, show=False)
            plt.title('SHAP Feature Importance Summary', fontweight='bold')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
            self.save_modern_plot(fig, '15_shap_summary', '')
            return fig
        except Exception as e:
            print(f"  SHAP summary failed: {e}")
            return None

    # ==================== 4. HYPERPARAMETER & VALIDATION PLOTS ====================

    def plot_validation_curve(self, param_name='batch_size', param_range=[16,32,64,128], figsize=(10, 6)):
        """
        Validation curve for a hyperparameter.
        For NN, this will be slow as it retrains the model for each parameter.
        Consider using it only on small subsets.
        """
        try:
            from tensorflow.keras.models import clone_model
            train_scores, test_scores = validation_curve(
                self.model, self.X_train[:500], self.y_train[:500],
                param_name=param_name, param_range=param_range,
                cv=3, scoring='r2', n_jobs=-1
            )
            fig, ax = plt.subplots(figsize=figsize)
            train_mean = np.mean(train_scores, axis=1)
            test_mean = np.mean(test_scores, axis=1)
            train_std = np.std(train_scores, axis=1)
            test_std = np.std(test_scores, axis=1)

            ax.fill_between(param_range, train_mean - train_std, train_mean + train_std,
                            alpha=0.2, color=self.colors['primary'])
            ax.fill_between(param_range, test_mean - test_std, test_mean + test_std,
                            alpha=0.2, color=self.colors['secondary'])
            ax.plot(param_range, train_mean, 'o-', color=self.colors['primary'], label='Train')
            ax.plot(param_range, test_mean, 'o-', color=self.colors['secondary'], label='Validation')
            ax.set_xlabel(param_name)
            ax.set_ylabel('R² Score')
            ax.set_title(f'Validation Curve: {param_name}', fontweight='bold')
            ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
            self.save_modern_plot(fig, '16_validation_curve', '')
            return fig
        except Exception as e:
            print(f"  Validation curve failed for NN: {e}")
            return None

    def plot_cross_validation_scores(self, cv=5, figsize=(10, 6)):
        """Cross‑validation R² scores using sklearn's cross_val_score."""
        try:
            # For NN, this will refit the model many times – may be slow.
            scores = cross_val_score(self.model, self.X_train, self.y_train, cv=cv, scoring='r2', n_jobs=-1)
            fig, ax = plt.subplots(figsize=figsize)
            x = np.arange(1, cv+1)
            bars = ax.bar(x, scores, color=plt.cm.viridis(np.linspace(0.2, 0.8, cv)), edgecolor='white')
            for bar, score in zip(bars, scores):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{score:.3f}', ha='center', va='bottom', fontsize=9)
            ax.axhline(y=np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.4f}')
            ax.set_xlabel('Fold')
            ax.set_ylabel('R² Score')
            ax.set_title(f'{cv}-Fold Cross-Validation', fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([f'Fold {i}' for i in x])
            ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
            ax.grid(True, alpha=0.1, axis='y')
            self._remove_spines(ax)
            self.save_modern_plot(fig, '17_cross_validation', '')
            return fig
        except Exception as e:
            print(f"  Cross‑validation plot failed for NN: {e}")
            return None

    # ==================== 5. NN-SPECIFIC PLOTS (training history, learning rate) ====================

    def plot_training_history(self, figsize=(10, 6)):
        """Plot training and validation loss over epochs (from history)"""
        if self.history is None:
            print("  No training history provided. Skipping training history plot.")
            return None
    
        fig, ax = plt.subplots(figsize=figsize)
        
        # Handle both History object and plain dict
        if hasattr(self.history, 'history'):
            history_dict = self.history.history
        else:
            history_dict = self.history   # already a dict
    
        # Plot loss
        if 'loss' in history_dict:
            ax.plot(history_dict['loss'], label='Training Loss',
                    color=self.gradients['train'][0], linewidth=2.5, alpha=0.9)
        if 'val_loss' in history_dict:
            ax.plot(history_dict['val_loss'], label='Validation Loss',
                    color=self.gradients['test'][0], linewidth=2.5, alpha=0.9)
    
        ax.set_xlabel('Epoch', fontweight='bold')
        ax.set_ylabel('Loss (MAE)', fontweight='bold')
        ax.set_title('Training History – Loss Convergence', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
    
        # Best epoch annotation
        if 'val_loss' in history_dict:
            best_epoch = np.argmin(history_dict['val_loss'])
            best_val_loss = history_dict['val_loss'][best_epoch]
            ax.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.7,
                       label=f'Best epoch: {best_epoch}, val_loss={best_val_loss:.4f}')
            ax.legend()
    
        # Add final loss text
        final_train = history_dict['loss'][-1] if 'loss' in history_dict else None
        final_val = history_dict['val_loss'][-1] if 'val_loss' in history_dict else None
        text = f"Final Loss\nTrain: {final_train:.4f}\nVal: {final_val:.4f}" if final_val else f"Final Train Loss: {final_train:.4f}"
        self.create_modern_textbox(ax, text, 'upper right', fontsize=10, border=True)
    
        self._remove_spines(ax)
        self.save_modern_plot(fig, '18_training_history', '')
        return fig

    def plot_learning_rate_schedule(self, figsize=(10, 6)):
        """Plot learning rate schedule from history if available"""
        if self.history is None:
            print("  No training history provided. Skipping learning rate plot.")
            return None
    
        fig, ax = plt.subplots(figsize=figsize)
        
        # Handle both History object and plain dict
        if hasattr(self.history, 'history'):
            history_dict = self.history.history
        else:
            history_dict = self.history
    
        if 'lr' in history_dict:
            ax.semilogy(history_dict['lr'], color=self.colors['purple'], linewidth=2.5, alpha=0.9)
            ax.set_xlabel('Epoch', fontweight='bold')
            ax.set_ylabel('Learning Rate', fontweight='bold')
            ax.set_title('Learning Rate Schedule', fontweight='bold')
            ax.grid(True, alpha=0.1)
    
            # Add stats
            initial_lr = history_dict['lr'][0]
            final_lr = history_dict['lr'][-1]
            text = f"Initial LR: {initial_lr:.2e}\nFinal LR: {final_lr:.2e}"
            self.create_modern_textbox(ax, text, 'upper right', fontsize=10, border=True)
        else:
            ax.text(0.5, 0.5, 'Learning rate history not available', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12)
            ax.set_title('Learning Rate Schedule', fontweight='bold')
            ax.axis('off')
    
        self._remove_spines(ax)
        self.save_modern_plot(fig, '19_learning_rate_schedule', '')
        return fig

    # ==================== 6. ORIGINAL LIGHTGBM SPECIALTY PLOTS (adapted for NN) ====================

    def plot_performance_metrics(self, figsize=(12, 6)):
        """Bar chart comparing train/test metrics"""
        metrics = {
            'R²': [self.metrics['train']['R2'], self.metrics['test']['R2']],
            'RMSE': [self.metrics['train']['RMSE'], self.metrics['test']['RMSE']],
            'MAE': [self.metrics['train']['MAE'], self.metrics['test']['MAE']],
            'MAPE': [self.metrics['train']['MAPE'], self.metrics['test']['MAPE']]
        }
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

        metric_names = list(metrics.keys())
        train_values = [metrics[m][0] for m in metric_names]
        test_values = [metrics[m][1] for m in metric_names]
        x = np.arange(len(metric_names))
        width = 0.35

        ax1.bar(x - width/2, train_values, width, label='Training',
                color=self.gradients['train'][0], alpha=0.8, edgecolor='white')
        ax1.bar(x + width/2, test_values, width, label='Test',
                color=self.gradients['test'][0], alpha=0.8, edgecolor='white')
        ax1.set_xlabel('Metrics', fontweight='bold')
        ax1.set_ylabel('Value', fontweight='bold')
        ax1.set_title('Performance Metrics Comparison', fontweight='bold')
        ax1.set_xticks(x)
        ax1.set_xticklabels(metric_names)
        ax1.legend()
        ax1.grid(True, alpha=0.1, axis='y')

        for i, (train_val, test_val) in enumerate(zip(train_values, test_values)):
            ax1.text(i - width/2, train_val + 0.01, f'{train_val:.4f}',
                     ha='center', va='bottom', fontsize=8)
            ax1.text(i + width/2, test_val + 0.01, f'{test_val:.4f}',
                     ha='center', va='bottom', fontsize=8)

        ax2.axis('off')
        summary_text = (f"Overfitting Gap (ΔR²): {self.metrics['train']['R2'] - self.metrics['test']['R2']:.4f}\n"
                        f"Test R²: {self.metrics['test']['R2']:.4f}\n"
                        f"Test RMSE: {self.metrics['test']['RMSE']:.4f}\n"
                        f"Test MAE: {self.metrics['test']['MAE']:.4f}\n"
                        f"Test MAPE: {self.metrics['test']['MAPE']:.2f}%")
        self.create_modern_textbox(ax2, summary_text, 'center', fontsize=11, border=True)

        self.save_modern_plot(fig, '20_performance_metrics', '')
        return fig

    def plot_prediction_distributions(self, figsize=(7, 6)):
        """Actual vs predicted distribution for train and test"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        ax = axes[0]
        ax.hist(self.y_train, bins=30, alpha=0.5, color=self.gradients['train'][0],
                label='Actual', density=True, edgecolor='white')
        ax.hist(self.y_pred_train, bins=30, alpha=0.5, color=self.gradients['test'][0],
                label='Predicted', density=True, edgecolor='white')
        ax.set_xlabel('Target', fontweight='bold')
        ax.set_ylabel('Density', fontweight='bold')
        ax.set_title('Training Set Distribution', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.1)

        ax = axes[1]
        ax.hist(self.y_test, bins=30, alpha=0.5, color=self.gradients['train'][0],
                label='Actual', density=True, edgecolor='white')
        ax.hist(self.y_pred_test, bins=30, alpha=0.5, color=self.gradients['test'][0],
                label='Predicted', density=True, edgecolor='white')
        ax.set_xlabel('Target', fontweight='bold')
        ax.set_ylabel('Density', fontweight='bold')
        ax.set_title('Test Set Distribution', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.1)

        self.save_modern_plot(fig, '21_prediction_distributions', '')
        return fig

    def plot_model_parameters(self, figsize=(10, 8)):
        """Text summary of model architecture and parameters"""
        fig, ax = plt.subplots(figsize=figsize)

        # Count total parameters
        total_params = self.model.count_params()
        trainable_params = sum([np.prod(w.shape) for w in self.model.trainable_weights])
        non_trainable = total_params - trainable_params

        summary_text = "🧠 NEURAL NETWORK ARCHITECTURE\n" + "="*40 + "\n\n"
        summary_text += f"Total parameters: {total_params:,}\n"
        summary_text += f"Trainable parameters: {trainable_params:,}\n"
        summary_text += f"Non‑trainable parameters: {non_trainable:,}\n\n"
        summary_text += "Layer Summary:\n"
        for i, layer in enumerate(self.model.layers):
            summary_text += f"  {i+1}. {layer.name} ({layer.__class__.__name__}) – {layer.count_params():,} params\n"
            if hasattr(layer, 'units'):
                summary_text += f"     Units: {layer.units}\n"
            if hasattr(layer, 'activation'):
                summary_text += f"     Activation: {layer.activation.__name__ if hasattr(layer.activation, '__name__') else str(layer.activation)}\n"

        ax.axis('off')
        ax.text(0.02, 0.98, summary_text, va='top', ha='left',
                fontsize=9, fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=1", facecolor='white', alpha=0.9))
        self.save_modern_plot(fig, '22_model_parameters', '')
        return fig

    def plot_error_by_range(self, figsize=(12, 6)):
        """Error analysis across value bins"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        n_bins = 10
        bins = np.linspace(self.y_test.min(), self.y_test.max(), n_bins + 1)
        bin_stats = []
        for i in range(n_bins):
            mask = (self.y_test >= bins[i]) & (self.y_test < bins[i+1])
            if np.sum(mask) > 5:
                bin_center = (bins[i] + bins[i+1]) / 2
                mean_error = np.mean(self.residuals_test[mask])
                std_error = np.std(self.residuals_test[mask])
                n_samples = np.sum(mask)
                bin_stats.append({'center': bin_center, 'mean_error': mean_error,
                                  'std_error': std_error, 'n_samples': n_samples,
                                  'width': bins[i+1] - bins[i]})

        if bin_stats:
            centers = [s['center'] for s in bin_stats]
            mean_errors = [s['mean_error'] for s in bin_stats]
            std_errors = [s['std_error'] for s in bin_stats]
            widths = [s['width'] * 0.8 for s in bin_stats]

            ax1.bar(centers, mean_errors, width=widths, alpha=0.7,
                    color=self.colors['primary'], edgecolor='white', yerr=std_errors, capsize=5)
            ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
            ax1.set_xlabel('Actual Value Bins', fontweight='bold')
            ax1.set_ylabel('Mean Error ± Std', fontweight='bold')
            ax1.set_title('Mean Error by Value Range', fontweight='bold')
            ax1.grid(True, alpha=0.1)

            samples = [s['n_samples'] for s in bin_stats]
            ax2.bar(centers, samples, width=widths, alpha=0.7,
                    color=self.colors['accent'], edgecolor='white')
            ax2.set_xlabel('Actual Value Bins', fontweight='bold')
            ax2.set_ylabel('Number of Samples', fontweight='bold')
            ax2.set_title('Sample Distribution by Value Range', fontweight='bold')
            ax2.grid(True, alpha=0.1)
            self.create_modern_textbox(ax2, f'Total Samples: {np.sum(samples)}', 'upper right', fontsize=10)
        else:
            ax1.text(0.5, 0.5, 'Insufficient data for bins', ha='center', va='center')
            ax1.set_title('Error by Range', fontweight='bold')
            ax1.axis('off')
            ax2.axis('off')

        self.save_modern_plot(fig, '23_error_by_range', '')
        return fig

    def plot_confidence_intervals(self, figsize=(12, 6)):
        """Prediction intervals via bootstrap (residual bootstrap)"""
        n_bootstrap = 100
        n_samples = min(100, len(self.X_test))
        indices = np.random.choice(len(self.X_test), n_samples, replace=False)
        X_sample = self.X_test[indices]
        y_sample = self.y_test[indices]
        y_pred_sample = self.y_pred_test[indices]

        # Residual bootstrap
        residuals = self.residuals_test
        all_preds = []
        for _ in range(n_bootstrap):
            resampled_resid = np.random.choice(residuals, size=n_samples, replace=True)
            boot_pred = y_pred_sample + resampled_resid
            all_preds.append(boot_pred)
        all_preds = np.array(all_preds)

        pred_mean = all_preds.mean(axis=0)
        pred_lower = np.percentile(all_preds, 2.5, axis=0)
        pred_upper = np.percentile(all_preds, 97.5, axis=0)

        fig, ax = plt.subplots(figsize=figsize)
        sorted_idx = np.argsort(y_sample)
        x = np.arange(len(sorted_idx))
        ax.fill_between(x, pred_lower[sorted_idx], pred_upper[sorted_idx],
                        alpha=0.3, color=self.colors['primary'], label='95% CI')
        ax.plot(x, y_sample[sorted_idx], 'o', color=self.colors['danger'], markersize=6, label='Actual')
        ax.plot(x, pred_mean[sorted_idx], '-', color=self.colors['primary'], linewidth=2, label='Mean Prediction')

        coverage = np.mean((y_sample >= pred_lower) & (y_sample <= pred_upper)) * 100
        ax.set_title(f'Prediction Intervals - Coverage: {coverage:.1f}%', fontweight='bold')
        ax.set_xlabel('Sample (sorted)')
        ax.set_ylabel('Target')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_modern_plot(fig, '24_confidence_intervals', '')
        return fig

    def plot_model_complexity(self, figsize=(12, 6)):
        """Model complexity analysis – parameter counts and layer info"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

        # Bar chart of trainable vs non-trainable params
        trainable = sum([np.prod(w.shape) for w in self.model.trainable_weights])
        non_trainable = self.model.count_params() - trainable
        categories = ['Trainable', 'Non-trainable']
        values = [trainable, non_trainable]
        colors_bar = [self.colors['primary'], self.colors['secondary']]
        ax1.bar(categories, values, color=colors_bar, alpha=0.8, edgecolor='white')
        ax1.set_ylabel('Number of Parameters', fontweight='bold')
        ax1.set_title('Parameter Distribution', fontweight='bold')
        ax1.grid(True, alpha=0.1, axis='y')
        for i, v in enumerate(values):
            ax1.text(i, v + 0.01 * max(values), f'{v:,}', ha='center', va='bottom', fontsize=10)

        # Text summary
        ax2.axis('off')
        stats_text = "📊 MODEL COMPLEXITY STATS\n" + "="*30 + "\n\n"
        stats_text += f"Total layers: {len(self.model.layers)}\n"
        stats_text += f"Total parameters: {self.model.count_params():,}\n"
        stats_text += f"Trainable: {trainable:,}\n"
        stats_text += f"Non-trainable: {non_trainable:,}\n\n"
        stats_text += "Layer types:\n"
        from collections import Counter
        layer_types = Counter([layer.__class__.__name__ for layer in self.model.layers])
        for typ, count in layer_types.items():
            stats_text += f"  {typ}: {count}\n"

        ax2.text(0.5, 0.5, stats_text, ha='center', va='center',
                 fontsize=10, bbox=dict(boxstyle="round,pad=1", facecolor='white', alpha=0.9))
        self.save_modern_plot(fig, '25_model_complexity', '')
        return fig

    def plot_model_dashboard(self, figsize=(16, 12)):
        """Comprehensive diagnostic dashboard (adapted for NN)"""
        fig = plt.figure(figsize=figsize)
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
        # 1. Main Performance Summary
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.axis('off')
        overfitting_gap = self.metrics['train']['R2'] - self.metrics['test']['R2']
        summary_text = (f"🏆 MODEL PERFORMANCE\n" + "="*25 + "\n\n"
                        f"Train R²: {self.metrics['train']['R2']:.4f}\n"
                        f"Test R²:  {self.metrics['test']['R2']:.4f}\n"
                        f"ΔR²:      {overfitting_gap:.4f}\n"
                        f"RMSE:     {self.metrics['test']['RMSE']:.4f}\n"
                        f"MAE:      {self.metrics['test']['MAE']:.4f}\n"
                        f"Bias:     {np.mean(self.residuals_test):.4f}\n"
                        f"Std Err:  {np.std(self.residuals_test):.4f}")
        bg_color = 'lightgreen' if overfitting_gap < 0.1 else 'lightcoral'
        ax1.text(0.5, 0.5, summary_text, ha='center', va='center',
                 fontsize=11, bbox=dict(boxstyle="round,pad=1", facecolor=bg_color, alpha=0.8))
    
        # 2. Actual vs Predicted (mini)
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.scatter(self.y_test, self.y_pred_test, alpha=0.6, s=20, color=self.gradients['test'][0])
        ax2.plot([self.y_test.min(), self.y_test.max()], [self.y_test.min(), self.y_test.max()], 'r--', alpha=0.7)
        ax2.set_xlabel('Actual', fontsize=9)
        ax2.set_ylabel('Predicted', fontsize=9)
        ax2.set_title('Actual vs Predicted', fontweight='bold', fontsize=10)
        ax2.grid(True, alpha=0.1)
    
        # 3. Residuals Distribution (mini)
        ax3 = fig.add_subplot(gs[0, 2])
        ax3.hist(self.residuals_test, bins=20, alpha=0.7, color=self.gradients['residual'][0], edgecolor='white')
        ax3.axvline(x=0, color='red', linestyle='--', linewidth=1)
        ax3.set_xlabel('Residuals', fontsize=9)
        ax3.set_ylabel('Frequency', fontsize=9)
        ax3.set_title('Residual Distribution', fontweight='bold', fontsize=10)
        ax3.grid(True, alpha=0.1)
    
        # 4. Feature Importance (top 5) – using permutation importance
        ax4 = fig.add_subplot(gs[1, 0])
        try:
            perm = permutation_importance(self.model, self.X_test, self.y_test, n_repeats=5, random_state=42)
            importance = perm.importances_mean
            top_n = min(5, len(importance))
            top_idx = np.argsort(importance)[-top_n:][::-1]
            top_features = [self.feature_names[i] for i in top_idx]
            top_importance = importance[top_idx]
            y_pos = np.arange(top_n)
            ax4.barh(y_pos, top_importance, color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)), alpha=0.8)
            ax4.set_yticks(y_pos)
            ax4.set_yticklabels([self.get_feature_name(f) for f in top_features], fontsize=9)
            ax4.invert_yaxis()
            ax4.set_xlabel('Importance (Δ R²)', fontsize=9)
            ax4.set_title(f'Top {top_n} Features', fontweight='bold', fontsize=10)
            ax4.grid(True, alpha=0.1, axis='x')
        except Exception as e:
            ax4.text(0.5, 0.5, 'Importance not available', ha='center', va='center')
    
        # 5. Training History (mini) – handle both History object and dict
        ax5 = fig.add_subplot(gs[1, 1])
        history_dict = None
        if self.history is not None:
            # Check if it's a History object (has .history) or a dict
            if hasattr(self.history, 'history'):
                history_dict = self.history.history
            else:
                history_dict = self.history
    
        if history_dict is not None and 'loss' in history_dict:
            ax5.plot(history_dict['loss'], linewidth=1.5, color=self.colors['primary'], label='Train')
            if 'val_loss' in history_dict:
                ax5.plot(history_dict['val_loss'], linewidth=1.5, color=self.colors['secondary'], label='Val')
                ax5.legend(fontsize=7)
            ax5.set_xlabel('Epoch', fontsize=9)
            ax5.set_ylabel('Loss', fontsize=9)
            ax5.set_title('Training History', fontweight='bold', fontsize=10)
            ax5.grid(True, alpha=0.1)
        else:
            ax5.text(0.5, 0.5, 'No training history', ha='center', va='center', fontsize=10)
            ax5.set_title('Training History', fontweight='bold', fontsize=10)
            ax5.axis('off')
    
        # 6. Error Statistics
        ax6 = fig.add_subplot(gs[1, 2])
        ax6.axis('off')
        abs_errors = np.abs(self.residuals_test)
        error_stats = {'Mean': np.mean(abs_errors), 'Std': np.std(abs_errors),
                       'Max': np.max(abs_errors), '95th %ile': np.percentile(abs_errors, 95)}
        error_text = "📊 ERROR STATISTICS\n" + "="*20 + "\n\n"
        for stat, value in error_stats.items():
            error_text += f"{stat}: {value:.4f}\n"
        ax6.text(0.5, 0.5, error_text, ha='center', va='center',
                 fontsize=10, bbox=dict(boxstyle="round,pad=1", facecolor='lightblue', alpha=0.8))
    
        # 7. Data Statistics
        ax7 = fig.add_subplot(gs[2, 0])
        ax7.axis('off')
        data_text = (f"📈 DATA STATISTICS\n" + "="*20 + "\n\n"
                     f"Train Samples: {len(self.X_train)}\n"
                     f"Test Samples:  {len(self.X_test)}\n"
                     f"Features:      {self.X_train.shape[1]}\n"
                     f"Target Range:  [{self.y_test.min():.2f}, {self.y_test.max():.2f}]\n"
                     f"Target Mean:   {self.y_test.mean():.2f}")
        ax7.text(0.5, 0.5, data_text, ha='center', va='center',
                 fontsize=10, bbox=dict(boxstyle="round,pad=1", facecolor='lightyellow', alpha=0.8))
    
        # 8. Model Architecture Summary
        ax8 = fig.add_subplot(gs[2, 1])
        ax8.axis('off')
        total_params = self.model.count_params()
        arch_text = f"⚙️ MODEL ARCHITECTURE\n" + "="*20 + "\n\n"
        arch_text += f"Layers: {len(self.model.layers)}\n"
        arch_text += f"Total params: {total_params:,}\n"
        arch_text += f"Trainable: {total_params - self.model.count_params():,}\n"
        arch_text += f"Input shape: {self.model.input_shape}\n"
        arch_text += f"Output shape: {self.model.output_shape}"
        ax8.text(0.5, 0.5, arch_text, ha='center', va='center',
                 fontsize=10, bbox=dict(boxstyle="round,pad=1", facecolor='lavender', alpha=0.8))
    
        # 9. Model Assessment
        ax9 = fig.add_subplot(gs[2, 2])
        ax9.axis('off')
        assessment = []
        if overfitting_gap < 0.05:
            assessment.append("✅ Excellent generalization")
        elif overfitting_gap < 0.1:
            assessment.append("✓ Good generalization")
        elif overfitting_gap < 0.15:
            assessment.append("⚠️ Moderate overfitting")
        else:
            assessment.append("❌ Significant overfitting")
    
        if self.metrics['test']['R2'] > 0.9:
            assessment.append("✅ Excellent predictive power")
        elif self.metrics['test']['R2'] > 0.8:
            assessment.append("✓ Very good predictive power")
        elif self.metrics['test']['R2'] > 0.7:
            assessment.append("✓ Good predictive power")
        elif self.metrics['test']['R2'] > 0.6:
            assessment.append("⚠️ Moderate predictive power")
        else:
            assessment.append("❌ Low predictive power")
    
        bias = np.mean(self.residuals_test)
        if np.abs(bias) < 0.01 * np.mean(self.y_test):
            assessment.append("✅ Minimal bias")
        else:
            assessment.append(f"⚠️ Bias: {bias:.4f}")
    
        assessment_text = "🔍 MODEL ASSESSMENT\n" + "="*20 + "\n\n" + "\n".join(assessment)
        ax9.text(0.5, 0.5, assessment_text, ha='center', va='center',
                 fontsize=10, bbox=dict(boxstyle="round,pad=1", facecolor='mistyrose', alpha=0.8))
    
        fig.suptitle('NEURAL NETWORK MODEL DIAGNOSTIC DASHBOARD', fontsize=18, fontweight='bold', y=0.98)
        self.save_modern_plot(fig, '26_model_dashboard', '')
        return fig

    def plot_bootstrapped_metrics(self, n_bootstrap=200, figsize=(12, 6)):
        """Bootstrap R² distribution with Q-Q plot"""
        n = len(self.y_test)
        bootstrap_r2 = []
        for _ in range(n_bootstrap):
            idx = np.random.choice(n, n, replace=True)
            y_true = self.y_test[idx]
            y_pred = self.y_pred_test[idx]
            bootstrap_r2.append(r2_score(y_true, y_pred))
        bootstrap_r2 = np.array(bootstrap_r2)

        fig, axes = plt.subplots(1, 2, figsize=figsize)

        ax = axes[0]
        sns.histplot(bootstrap_r2, kde=True, color=self.colors['primary'], ax=ax, bins=30)
        ax.axvline(self.metrics['test']['R2'], color='red', linestyle='--', linewidth=2,
                   label=f'Observed: {self.metrics["test"]["R2"]:.4f}')
        ax.axvline(np.percentile(bootstrap_r2, 2.5), color='green', linestyle=':', linewidth=1.5)
        ax.axvline(np.percentile(bootstrap_r2, 97.5), color='green', linestyle=':', linewidth=1.5)
        ax.set_xlabel('R² Score')
        ax.set_title('Bootstrap R² Distribution', fontweight='bold')
        ax.legend(framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        ax = axes[1]
        stats.probplot(bootstrap_r2, dist="norm", plot=ax)
        ax.get_lines()[0].set_marker('o')
        ax.get_lines()[0].set_markersize(4)
        ax.get_lines()[1].set_color('red')
        ax.set_title('Q-Q Plot (Normality Check)', fontweight='bold')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)

        fig.suptitle('Model Stability via Bootstrap', fontweight='bold')
        self.save_modern_plot(fig, '27_bootstrapped_metrics', '')
        return fig

    # ==================== GENERATE ALL PLOTS ====================

    def generate_all_plots(self):
        """Generate all modern diagnostic plots for Neural Network"""
        print("=" * 70)
        print("🎨 Neural Network Modern Visualization Suite")
        print("=" * 70)

        count = 0
        print("\n📊 EDA Plots:")
        self.plot_correlation_heatmap(); count += 1
        self.plot_distributions(); count += 1
        self.plot_boxplots(); count += 1
        self.plot_feature_scatter(); count += 1
        self.plot_missing_values(); count += 1
        self.plot_outlier_detection(); count += 1

        print("\n📈 Performance & Residuals:")
        self.plot_actual_vs_predicted(); count += 1
        self.plot_residuals_vs_predicted(); count += 1
        self.plot_residual_distribution(); count += 1
        self.plot_learning_curves(); count += 1
        self.plot_cumulative_error(); count += 1
        self.plot_performance_metrics(); count += 1
        self.plot_prediction_distributions(); count += 1

        print("\n🔍 Feature Importance:")
        self.plot_top_features(); count += 1
        self.plot_cumulative_features(); count += 1
        self.plot_permutation_importance(); count += 1
        # self.plot_shap_summary(); count += 1

        print("\n⚙️ Training & Architecture:")
        self.plot_training_history(); count += 1
        self.plot_learning_rate_schedule(); count += 1
        self.plot_model_parameters(); count += 1
        self.plot_model_complexity(); count += 1

        print("\n📐 Statistical Validation:")
        self.plot_confidence_intervals(); count += 1
        self.plot_bootstrapped_metrics(); count += 1

        print("\n📊 Advanced Diagnostics:")
        self.plot_error_by_range(); count += 1
        self.plot_model_dashboard(); count += 1

        # Optional hyperparameter plots (may be slow)
        # print("\n⚙️ Hyperparameter Validation (may be slow):")
        # self.plot_validation_curve()
        # self.plot_cross_validation_scores()

        print("\n" + "=" * 70)
        print(f"✅ Generated {count} plots in: {self.results_path}")
        print("=" * 70)
        return count

# MUltiple Plots

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.offsetbox import AnchoredText
import seaborn as sns
from scipy import stats
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    LOWESS_AVAILABLE = True
except ImportError:
    LOWESS_AVAILABLE = False

try:
    from scipy.stats import friedmanchisquare
    FRIEDMAN_AVAILABLE = True
except ImportError:
    FRIEDMAN_AVAILABLE = False

# Optional: for neural network specific visualisations
try:
    import tensorflow as tf
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False


class MultiModelNNComparator:
    """
    Compare multiple Neural Network models using absolute (raw) metric values.
    All transformations (normalization, inversion) have been removed.

    Expects visualizer_data from DataExtractionFrom_NNModernVisualizer_touseinMultiplePlots()
    which contains keys: 'metrics', 'y_pred_train', 'y_pred_test', 'residuals_train',
    'residuals_test', 'X_train', 'X_test', 'y_train', 'y_test', 'feature_names', 'model',
    and optionally 'history' (training loss/val loss) and 'weights'.
    """

    def __init__(self, visualizer_data, results_path="./multi_nn_comparison"):
        """
        Parameters
        ----------
        visualizer_data : dict {set_id: dict}
            Must contain keys: 'metrics', 'y_pred_train', 'y_pred_test',
            'residuals_train', 'residuals_test', 'X_train', 'X_test',
            'y_train', 'y_test', 'feature_names', 'model'
        results_path : str
        """
        self.results_path = results_path
        os.makedirs(results_path, exist_ok=True)

        self.visualizer_data = visualizer_data
        self.set_ids = list(visualizer_data.keys())

        # Extract data into internal structures
        self.models = {}
        self.histories = {}  # training history (if available)
        self.model_feature_names = {}
        self.data = {}
        self.predictions = {}
        self.metrics = {}

        for set_id, data in visualizer_data.items():
            self.models[set_id] = data['model']
            self.model_feature_names[set_id] = data['feature_names']
            self.data[set_id] = {
                'X_train': data['X_train'],
                'X_test': data['X_test'],
                'y_train': data['y_train'],
                'y_test': data['y_test']
            }
            self.predictions[set_id] = {
                'y_pred_train': data['y_pred_train'],
                'y_pred_test': data['y_pred_test'],
                'residuals_train': data['residuals_train'],
                'residuals_test': data['residuals_test']
            }
            self.metrics[set_id] = data['metrics']   # nested {'train': {...}, 'test': {...}}
            if 'history' in data:
                self.histories[set_id] = data['history']

        # Colour scheme (consistent with LightGBM)
        self.colors = {
            'primary': '#FF00FF', 'secondary': '#7FFF00', 'accent': '#008B8B',
            'danger': '#FFBF00', 'warning': '#9370DB', 'info': '#00FFFF',
            'dark': '#1F51FF', 'light': '#FFFFFF', 'success': '#4B0082', 'purple': '#FF073A',
        }
        self.model_colors = {sid: list(self.colors.values())[i % len(self.colors)]
                             for i, sid in enumerate(self.set_ids)}

        self.set_modern_style()
        self._print_metrics_summary()

    # ---------- Helper methods (identical to LightGBM version) ----------
    def set_modern_style(self):
        plt.style.use('default')
        mpl.rcParams.update({
            'font.family': 'sans-serif', 'font.sans-serif': ['Times New Roman', 'DejaVu Sans'],
            'font.size': 11, 'figure.dpi': 300, 'figure.figsize': (8, 6),
            'figure.titlesize': 16, 'figure.titleweight': 'bold',
            'axes.labelsize': 13, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
            'axes.linewidth': 1.2, 'axes.grid': True, 'grid.alpha': 0.15,
            'grid.linestyle': '-', 'grid.linewidth': 0.8, 'xtick.labelsize': 11,
            'ytick.labelsize': 11, 'legend.fontsize': 11, 'legend.frameon': True,
            'legend.framealpha': 0.95, 'legend.edgecolor': '#E5E7EB',
            'lines.linewidth': 2.5, 'lines.markersize': 6,
        })

    def create_modern_textbox(self, ax, text, location='upper left', fontsize=10,
                              bg_color='white', alpha=0.95, border=False):
        text_box = AnchoredText(text, loc=location, frameon=border,
                               pad=0.6, borderpad=0.8,
                               prop=dict(fontsize=fontsize, bbox=dict(boxstyle="round,pad=0.4",
                                                                      facecolor=bg_color,
                                                                      alpha=alpha,
                                                                      edgecolor='#E5E7EB' if border else 'none')))
        ax.add_artist(text_box)

    def save_plot(self, fig, name):
        fig.tight_layout(pad=2.0)
        filepath = os.path.join(self.results_path, f"{name}.png")
        fig.savefig(filepath, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none', pad_inches=0.2)
        plt.close(fig)
        print(f"  ✓ Saved: {name}.png")

    def _remove_spines(self, ax):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1)
        ax.spines['bottom'].set_linewidth(1)

    def _print_metrics_summary(self):
        print("\n" + "="*70)
        print("📊 Neural Network Model Performance Summary (Test Set) – Absolute Values")
        print("="*70)
        df = pd.DataFrame(index=self.set_ids, columns=['R²', 'RMSE', 'MAE', 'MAPE', 'Expl. Var'])
        for sid in self.set_ids:
            df.loc[sid, 'R²'] = f"{self.metrics[sid]['test']['R2']:.4f}"
            df.loc[sid, 'RMSE'] = f"{self.metrics[sid]['test']['RMSE']:.4f}"
            df.loc[sid, 'MAE'] = f"{self.metrics[sid]['test']['MAE']:.4f}"
            df.loc[sid, 'MAPE'] = f"{self.metrics[sid]['test']['MAPE']:.4f}"
            df.loc[sid, 'Expl. Var'] = f"{self.metrics[sid]['test']['Explained Variance']:.4f}"
        print(df.to_string())
        print("="*70 + "\n")

    # ---------- Comparative plots (same as LightGBM, adapted for NN) ----------

    def plot_parallel_coordinates(self, figsize=(10, 6)):
        """Parallel coordinates plot using raw metric values."""
        metrics_list = ['R2', 'RMSE', 'MAE', 'MAPE', 'Explained Variance']
        rows = []
        for set_id in self.set_ids:
            row = {'Model': f"Model {set_id}"}
            for m in metrics_list:
                val = self.metrics[set_id]['test'][m]
                row[m] = val
            rows.append(row)
        df = pd.DataFrame(rows)
        df.set_index('Model', inplace=True)

        fig, ax = plt.subplots(figsize=figsize)
        from pandas.plotting import parallel_coordinates
        df_plot = df.reset_index()
        parallel_coordinates(df_plot, 'Model', ax=ax,
                             color=[self.model_colors[s] for s in self.set_ids],
                             alpha=0.8, linewidth=2)
        ax.set_title("Parallel Coordinates – NN Performance (Raw Values)", fontweight='bold')
        ax.set_ylabel("Metric Value (raw, different scales)")
        ax.set_xlabel("Metric")
        ax.grid(True, alpha=0.1)
        self.create_modern_textbox(ax, "Note: Different metrics have different scales.\nR² ~ 0.97, Error metrics ~ 0.001",
                                   location='lower left', fontsize=8, border=False)
        self._remove_spines(ax)
        self.save_plot(fig, "01_parallel_coordinates")

    def plot_metrics_bar_chart(self, figsize=(14, 10)):
        """Bar chart for each metric using raw values."""
        metrics_list = ['R2', 'RMSE', 'MAE', 'MAPE', 'Explained Variance']
        n_metrics = len(metrics_list)
        fig, axes = plt.subplots(1, n_metrics, figsize=figsize, sharey=False)
        if n_metrics == 1:
            axes = [axes]

        for idx, metric in enumerate(metrics_list):
            ax = axes[idx]
            values = []
            model_labels = []
            for set_id in self.set_ids:
                val = self.metrics[set_id]['test'][metric]
                values.append(val)
                model_labels.append(f"Model {set_id}")

            bars = ax.bar(model_labels, values, color=[self.model_colors[s] for s in self.set_ids],
                          edgecolor='white', linewidth=1.5, alpha=0.8)

            for bar, val in zip(bars, values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(values),
                        f'{val:.4f}', ha='center', va='bottom', fontsize=9)

            ax.set_title(metric, fontweight='bold', fontsize=12)
            ax.set_ylabel(metric if metric != 'Explained Variance' else 'Expl. Var', fontsize=10)
            ax.grid(True, alpha=0.1, axis='y')
            self._remove_spines(ax)

            y_min, y_max = ax.get_ylim()
            ax.set_ylim(y_min, y_max * 1.1)

        fig.suptitle("NN Model Performance Comparison – Raw Metric Values", fontweight='bold', fontsize=14, y=1.02)
        plt.tight_layout()
        self.save_plot(fig, "02_metrics_bar_chart")

    def plot_radar_chart(self, figsize=(8, 8)):
        """Radar chart using raw values."""
        metrics = ['R²', 'RMSE', 'MAE', 'MAPE', 'Expl. Var']
        values = {}
        for set_id in self.set_ids:
            vals = [
                self.metrics[set_id]['test']['R2'],
                self.metrics[set_id]['test']['RMSE'],
                self.metrics[set_id]['test']['MAE'],
                self.metrics[set_id]['test']['MAPE'],
                self.metrics[set_id]['test']['Explained Variance']
            ]
            values[set_id] = vals

        angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
        for i, set_id in enumerate(self.set_ids):
            raw_vals = values[set_id]
            raw_vals += raw_vals[:1]  # close loop
            ax.plot(angles, raw_vals, 'o-', linewidth=2,
                    color=self.model_colors[set_id], label=f"Model {set_id}")
            ax.fill(angles, raw_vals, alpha=0.1, color=self.model_colors[set_id])

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics, fontsize=10)
        ax.set_title("Radar Chart – NN Performance (Raw Values)", fontweight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
        self.create_modern_textbox(ax, "Note: R² ~ 0.97 dominates the scale.\nError metrics are near zero.",
                                   location='lower left', fontsize=8, border=False)
        self.save_plot(fig, "03_radar_chart")

    def plot_comparative_learning_curves(self, cv=5, figsize=(10, 7)):
        """
        Learning curves using cross-validation (works for any sklearn‑compatible estimator).
        For pure Keras models, we fall back to training history if available.
        """
        fig, ax = plt.subplots(figsize=figsize)

        # If histories exist, plot loss curves (more NN‑oriented)
        if self.histories:
            for set_id in self.set_ids:
                hist = self.histories[set_id]
                if 'loss' in hist and 'val_loss' in hist:
                    epochs = range(1, len(hist['loss']) + 1)
                    ax.plot(epochs, hist['loss'], '--', color=self.model_colors[set_id],
                            label=f"Model {set_id} (train loss)")
                    ax.plot(epochs, hist['val_loss'], '-', color=self.model_colors[set_id],
                            label=f"Model {set_id} (val loss)", linewidth=2)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.set_title("Training History (Loss) – Neural Networks", fontweight='bold')
            ax.legend(loc='upper right', framealpha=0.95)
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
            self.save_plot(fig, "04_training_history")
        else:
            # Fallback to sklearn learning_curve (R²)
            for set_id in self.set_ids:
                model = self.models[set_id]
                X = self.data[set_id]['X_train']
                y = self.data[set_id]['y_train']
                try:
                    train_sizes, train_scores, test_scores = learning_curve(
                        model, X, y, cv=cv, n_jobs=-1,
                        train_sizes=np.linspace(0.1, 1.0, 8), scoring='r2'
                    )
                    train_mean = np.mean(train_scores, axis=1)
                    test_mean = np.mean(test_scores, axis=1)
                    train_std = np.std(train_scores, axis=1)
                    test_std = np.std(test_scores, axis=1)
                    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                                    alpha=0.2, color=self.model_colors[set_id])
                    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                                    alpha=0.2, color=self.model_colors[set_id])
                    ax.plot(train_sizes, train_mean, '--', color=self.model_colors[set_id],
                            label=f"Model {set_id} (train)")
                    ax.plot(train_sizes, test_mean, '-', color=self.model_colors[set_id],
                            label=f"Model {set_id} (val)", linewidth=2)
                except Exception as e:
                    print(f"Learning curve for Model {set_id} failed: {e}")
            ax.set_xlabel("Training examples")
            ax.set_ylabel("R² score")
            ax.set_title("Comparative Learning Curves – NN", fontweight='bold')
            ax.legend(loc='lower right', ncol=2, framealpha=0.95)
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
            self.save_plot(fig, "04_comparative_learning_curves")

    def plot_residuals_with_loess(self, figsize=(10, 7)):
        """Residuals vs predicted with LOESS smoothing."""
        fig, ax = plt.subplots(figsize=figsize)
        for set_id in self.set_ids:
            y_pred = self.predictions[set_id]['y_pred_test']
            residuals = self.predictions[set_id]['residuals_test']
            ax.scatter(y_pred, residuals, alpha=0.4, s=20,
                      color=self.model_colors[set_id], edgecolors='white',
                      label=f"Model {set_id} residuals")
            if LOWESS_AVAILABLE:
                try:
                    sorted_idx = np.argsort(y_pred)
                    y_pred_sorted = y_pred[sorted_idx]
                    resid_sorted = residuals[sorted_idx]
                    smoothed = lowess(resid_sorted, y_pred_sorted, frac=0.3)
                    ax.plot(smoothed[:, 0], smoothed[:, 1], '-',
                            color=self.model_colors[set_id], linewidth=2.5,
                            label=f"LOESS (Model {set_id})")
                except Exception as e:
                    print(f"LOESS failed for Model {set_id}: {e}")
            else:
                window = max(5, int(len(y_pred) * 0.05))
                df = pd.DataFrame({'pred': y_pred, 'resid': residuals})
                df_sorted = df.sort_values('pred')
                ma = df_sorted['resid'].rolling(window, center=True).mean()
                ax.plot(df_sorted['pred'], ma, '-', color=self.model_colors[set_id],
                        linewidth=2, label=f"Moving avg (Model {set_id})")
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.set_xlabel("Predicted values")
        ax.set_ylabel("Residuals")
        ax.set_title("Residuals vs Predicted with LOESS – NN", fontweight='bold')
        ax.legend(loc='lower right', framealpha=0.95)
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_plot(fig, "05_residuals_loess")

    def plot_qq_plots(self, figsize=(10, 8)):
        """Single Q‑Q plot comparing standardized residuals of all models."""
        fig, ax = plt.subplots(figsize=figsize)
        all_quantiles = []

        for set_id in self.set_ids:
            residuals = self.predictions[set_id]['residuals_test']
            residuals_std = (residuals - np.mean(residuals)) / np.std(residuals)
            osm, osr = stats.probplot(residuals_std, dist="norm", fit=False)
            all_quantiles.extend(osm)
            all_quantiles.extend(osr)
            ax.plot(osm, osr, 'o', markersize=4,
                    markerfacecolor=self.model_colors[set_id],
                    alpha=0.7, label=f"Model {set_id}")

        min_val = min(all_quantiles)
        max_val = max(all_quantiles)
        ax.plot([min_val, max_val], [min_val, max_val], 'r--',
                linewidth=2, alpha=0.8, label='Normal reference (y=x)')

        ax.set_xlabel("Theoretical quantiles", fontweight='bold', fontsize=12)
        ax.set_ylabel("Sample quantiles (standardized residuals)", fontweight='bold', fontsize=12)
        ax.set_title("Q‑Q Plot of Standardized Residuals – NN Models", fontweight='bold', fontsize=14)
        ax.legend(loc='lower right', framealpha=0.95, edgecolor='#E5E7EB')
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_plot(fig, "06_qq_plots")

    def plot_cross_validation_comparison(self, metric='R2', figsize=(10, 5)):
        """Cross-validation boxplot comparison."""
        cv_scores = {}
        for set_id in self.set_ids:
            model = self.models[set_id]
            X = self.data[set_id]['X_train']
            y = self.data[set_id]['y_train']
            try:
                scores = cross_val_score(model, X, y, cv=5, scoring='r2')
                cv_scores[set_id] = scores
            except Exception as e:
                print(f"CV failed for Model {set_id}: {e}")
                cv_scores[set_id] = np.array([np.nan])
        fig, ax = plt.subplots(figsize=figsize)
        means = [np.nanmean(cv_scores[s]) for s in self.set_ids]
        stds = [np.nanstd(cv_scores[s]) for s in self.set_ids]
        x = np.arange(len(self.set_ids))
        ax.bar(x, means, yerr=stds, capsize=5, color=[self.model_colors[s] for s in self.set_ids],
               edgecolor='white', linewidth=1.5, alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels([f"Model {s}" for s in self.set_ids])
        ax.set_ylabel(f"Cross‑validated {metric}")
        ax.set_title(f"{metric} – 5‑fold CV (NN)", fontweight='bold')
        ax.grid(True, alpha=0.1, axis='y')
        self._remove_spines(ax)
        if FRIEDMAN_AVAILABLE and all(len(cv_scores[s]) > 1 for s in self.set_ids):
            try:
                stat, p = friedmanchisquare(*[cv_scores[s] for s in self.set_ids])
                text = f"Friedman test: p = {p:.4f}"
                if p < 0.05:
                    text += "\nSignificant difference (p<0.05)"
                self.create_modern_textbox(ax, text, 'upper right', fontsize=9, border=True)
            except Exception as e:
                print(f"Friedman test failed: {e}")
        self.save_plot(fig, "07_cross_validation_comparison")

    def plot_bootstrap_boxplots(self, n_bootstrap=500, figsize=(8, 6)):
        """Bootstrapped R² distribution boxplots."""
        fig, ax = plt.subplots(figsize=figsize)
        bootstrap_data = []
        labels = []
        for set_id in self.set_ids:
            y_true = self.data[set_id]['y_test']
            y_pred = self.predictions[set_id]['y_pred_test']
            n = len(y_true)
            boot_scores = []
            for _ in range(n_bootstrap):
                idx = np.random.choice(n, n, replace=True)
                boot_scores.append(r2_score(y_true[idx], y_pred[idx]))
            bootstrap_data.append(boot_scores)
            labels.append(f"Model {set_id}")
        bp = ax.boxplot(bootstrap_data, labels=labels, patch_artist=True,
                        boxprops=dict(linewidth=1.5), whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5), medianprops=dict(color='red', linewidth=2))
        for patch, set_id in zip(bp['boxes'], self.set_ids):
            patch.set_facecolor(self.model_colors[set_id])
            patch.set_alpha(0.7)
        ax.set_ylabel("R² score")
        ax.set_title(f"Bootstrapped R² Distribution (n={n_bootstrap}) – NN", fontweight='bold')
        ax.grid(True, alpha=0.1, axis='y')
        self._remove_spines(ax)
        self.save_plot(fig, "08_bootstrap_boxplots")

    def plot_prediction_intervals(self, n_bootstrap=500, sample_size=50, figsize=(12, 6)):
        """Prediction intervals via residual bootstrap (similar to LightGBM)."""
        fig, axes = plt.subplots(1, 3, figsize=figsize)
        for idx, set_id in enumerate(self.set_ids):
            ax = axes[idx]
            y_pred = self.predictions[set_id]['y_pred_test']
            residuals = self.predictions[set_id]['residuals_test']
            n_samples = min(sample_size, len(y_pred))
            indices = np.random.choice(len(y_pred), n_samples, replace=False)
            y_pred_sample = y_pred[indices]
            y_test_sample = self.data[set_id]['y_test'][indices]
            boot_preds = []
            for _ in range(n_bootstrap):
                resampled_resid = np.random.choice(residuals, size=n_samples, replace=True)
                boot_pred = y_pred_sample + resampled_resid
                boot_preds.append(boot_pred)
            boot_preds = np.array(boot_preds)
            pred_lower = np.percentile(boot_preds, 2.5, axis=0)
            pred_upper = np.percentile(boot_preds, 97.5, axis=0)
            sorted_idx = np.argsort(y_test_sample)
            x_vals = np.arange(len(sorted_idx))
            ax.fill_between(x_vals, pred_lower[sorted_idx], pred_upper[sorted_idx],
                            alpha=0.3, color=self.model_colors[set_id], label='95% CI')
            ax.plot(x_vals, y_test_sample[sorted_idx], 'o', color=self.colors['danger'],
                    markersize=4, label='Actual')
            ax.plot(x_vals, y_pred_sample[sorted_idx], '-', color=self.model_colors[set_id],
                    linewidth=2, label='Predicted')
            coverage = np.mean((y_test_sample >= pred_lower) & (y_test_sample <= pred_upper)) * 100
            ax.set_title(f"Model {set_id} – Coverage: {coverage:.1f}%", fontweight='bold')
            ax.set_xlabel("Sample (sorted by actual)")
            ax.set_ylabel("Target")
            ax.legend(fontsize=8, framealpha=0.95)
            ax.grid(True, alpha=0.1)
            self._remove_spines(ax)
        fig.suptitle("Prediction Intervals (Residual Bootstrap, 95% CI) – NN", fontweight='bold', y=1.02)
        self.save_plot(fig, "09_prediction_intervals")

    def plot_comparative_shap(self, sample_size=200, top_features=10, figsize=(15, 12)):
        """SHAP summary plots for each model (uses DeepExplainer for Keras, else KernelExplainer)."""
        if not SHAP_AVAILABLE:
            print("  SHAP not installed. Skipping comparative SHAP plots.")
            return
        fig, axes = plt.subplots(1, 3, figsize=figsize)
        for idx, set_id in enumerate(self.set_ids):
            ax = axes[idx]
            X_test = self.data[set_id]['X_test']
            model = self.models[set_id]
            n_samples = min(sample_size, len(X_test))
            indices = np.random.choice(len(X_test), n_samples, replace=False)
            X_sample = X_test[indices] if not isinstance(X_test, pd.DataFrame) else X_test.iloc[indices]
            feature_names = self.model_feature_names[set_id]
            try:
                # Try to use the appropriate explainer
                if TF_AVAILABLE and isinstance(model, tf.keras.Model):
                    # For Keras models, use DeepExplainer (needs a background dataset)
                    background = X_sample[:100]  # use first 100 as background
                    explainer = shap.DeepExplainer(model, background)
                else:
                    # Fallback to KernelExplainer (slower but works for any model)
                    explainer = shap.KernelExplainer(model.predict, X_sample[:100])
                shap_values = explainer.shap_values(X_sample)
                # If output is a list (multi-output), take first element
                if isinstance(shap_values, list):
                    shap_values = shap_values[0]
                shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                                  max_display=top_features, show=False, ax=ax)
                ax.set_title(f"Model {set_id} – SHAP Summary", fontweight='bold')
            except Exception as e:
                ax.text(0.5, 0.5, f"SHAP failed\n{str(e)[:80]}", ha='center', va='center')
                ax.set_title(f"Model {set_id}")
        fig.suptitle("Comparative SHAP Feature Importance – NN", fontweight='bold', y=1.02)
        self.save_plot(fig, "10_comparative_shap")

    def plot_feature_importance_comparison(self, top_n=10, figsize=(12, 10)):
        """Permutation importance comparison (works for any model)."""
        fig, axes = plt.subplots(1, 3, figsize=figsize)
        for idx, set_id in enumerate(self.set_ids):
            ax = axes[idx]
            model = self.models[set_id]
            X_test = self.data[set_id]['X_test']
            y_test = self.data[set_id]['y_test']
            try:
                perm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42)
                sorted_idx = perm.importances_mean.argsort()[::-1][:top_n]
                y_pos = np.arange(len(sorted_idx))
                means = perm.importances_mean[sorted_idx]
                stds = perm.importances_std[sorted_idx]
                feature_labels = [self.model_feature_names[set_id][i] for i in sorted_idx]
                ax.barh(y_pos, means, xerr=stds, capsize=3,
                        color=self.model_colors[set_id], edgecolor='white')
                ax.set_yticks(y_pos)
                ax.set_yticklabels(feature_labels)
                ax.invert_yaxis()
                ax.set_xlabel("Permutation Importance (Δ R²)")
                ax.set_title(f"Model {set_id}", fontweight='bold')
                ax.grid(True, alpha=0.1, axis='x')
                self._remove_spines(ax)
            except Exception as e:
                ax.text(0.5, 0.5, f"Failed: {e}", ha='center', va='center')
        fig.suptitle("Top Feature Importance – NN Comparison", fontweight='bold', y=1.02)
        self.save_plot(fig, "11_feature_importance_comparison")

    def plot_actual_vs_predicted_all(self, figsize=(10, 8)):
        """Scatter plot of actual vs predicted for all models on same axes."""
        fig, ax = plt.subplots(figsize=figsize)
        for set_id in self.set_ids:
            y_actual = self.data[set_id]['y_test']
            y_pred = self.predictions[set_id]['y_pred_test']
            r2_val = self.metrics[set_id]['test']['R2']
            ax.scatter(y_actual, y_pred, alpha=0.4, s=20,
                      color=self.model_colors[set_id],
                      label=f"Model {set_id} (R²={r2_val:.3f})")
        min_val = min(self.data[s]['y_test'].min() for s in self.set_ids)
        max_val = max(self.data[s]['y_test'].max() for s in self.set_ids)
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.6, label='Perfect')
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_title("Actual vs Predicted – All NN Models", fontweight='bold')
        ax.legend(loc='lower right', framealpha=0.95)
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_plot(fig, "12_actual_vs_predicted_all")

    # ---------- Additional neural‑network‑specific plots ----------

    def plot_residual_density_comparison(self, figsize=(10, 6)):
        """KDE plot of test residuals for all models."""
        fig, ax = plt.subplots(figsize=figsize)
        for set_id in self.set_ids:
            residuals = self.predictions[set_id]['residuals_test']
            sns.kdeplot(residuals, label=f"Model {set_id}", color=self.model_colors[set_id],
                        linewidth=2.5, ax=ax)
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.set_xlabel("Residuals")
        ax.set_ylabel("Density")
        ax.set_title("Residual Distribution (Test Set) – NN Models", fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.1)
        self._remove_spines(ax)
        self.save_plot(fig, "13_residual_density")

    def plot_train_test_metric_comparison(self, metric='R2', figsize=(8, 6)):
        """
        Bar plot comparing train vs test performance for each model.
        Useful to detect overfitting.
        """
        metrics_list = ['R2', 'RMSE', 'MAE', 'MAPE']
        n_metrics = len(metrics_list)
        fig, axes = plt.subplots(1, n_metrics, figsize=(12, 5))
        if n_metrics == 1:
            axes = [axes]

        for idx, metric_name in enumerate(metrics_list):
            ax = axes[idx]
            train_vals = [self.metrics[sid]['train'][metric_name] for sid in self.set_ids]
            test_vals = [self.metrics[sid]['test'][metric_name] for sid in self.set_ids]
            x = np.arange(len(self.set_ids))
            width = 0.35
            ax.bar(x - width/2, train_vals, width, label='Train',
                   color='#1f77b4', alpha=0.8, edgecolor='white')
            ax.bar(x + width/2, test_vals, width, label='Test',
                   color='#ff7f0e', alpha=0.8, edgecolor='white')
            ax.set_xticks(x)
            ax.set_xticklabels([f"Model {s}" for s in self.set_ids])
            ax.set_ylabel(metric_name)
            ax.set_title(f"{metric_name} – Train vs Test")
            ax.legend()
            ax.grid(True, alpha=0.1, axis='y')
            self._remove_spines(ax)
        fig.suptitle("Train/Test Performance Comparison – NN", fontweight='bold', y=1.02)
        self.save_plot(fig, "14_train_test_comparison")

    def plot_weight_histograms(self, figsize=(15, 10)):
        """
        Histograms of neural network weights (if model is Keras and weights accessible).
        """
        if not TF_AVAILABLE:
            print("  TensorFlow not available, skipping weight histograms.")
            return
        n_models = len(self.set_ids)
        fig, axes = plt.subplots(1, n_models, figsize=figsize)
        if n_models == 1:
            axes = [axes]
        for idx, set_id in enumerate(self.set_ids):
            ax = axes[idx]
            model = self.models[set_id]
            if isinstance(model, tf.keras.Model):
                all_weights = []
                for layer in model.layers:
                    weights = layer.get_weights()
                    for w in weights:
                        if w.size > 0:
                            all_weights.extend(w.flatten())
                if all_weights:
                    ax.hist(all_weights, bins=50, density=True, alpha=0.7,
                            color=self.model_colors[set_id], edgecolor='white')
                    ax.set_title(f"Model {set_id} – Weight Distribution")
                    ax.set_xlabel("Weight value")
                    ax.set_ylabel("Density")
                    ax.grid(True, alpha=0.1)
                    self._remove_spines(ax)
                else:
                    ax.text(0.5, 0.5, "No weights found", ha='center', va='center')
            else:
                ax.text(0.5, 0.5, "Model not a Keras instance", ha='center', va='center')
        fig.suptitle("Neural Network Weight Distributions", fontweight='bold', y=1.02)
        self.save_plot(fig, "15_weight_histograms")

    def plot_error_vs_feature(self, top_features=3, figsize=(15, 10)):
        """
        For each model, plot residuals against top 3 most important features (by permutation importance).
        Helps identify systematic errors.
        """
        # First, get permutation importance for each model to decide top features
        importance_dict = {}
        for set_id in self.set_ids:
            try:
                model = self.models[set_id]
                X_test = self.data[set_id]['X_test']
                y_test = self.data[set_id]['y_test']
                perm = permutation_importance(model, X_test, y_test, n_repeats=3, random_state=42)
                sorted_idx = perm.importances_mean.argsort()[::-1][:top_features]
                importance_dict[set_id] = sorted_idx
            except Exception:
                importance_dict[set_id] = list(range(min(top_features, len(self.model_feature_names[set_id]))))

        n_models = len(self.set_ids)
        fig, axes = plt.subplots(n_models, top_features, figsize=figsize)
        if n_models == 1 and top_features == 1:
            axes = np.array([[axes]])
        elif n_models == 1:
            axes = axes.reshape(1, -1)
        elif top_features == 1:
            axes = axes.reshape(-1, 1)

        for i, set_id in enumerate(self.set_ids):
            residuals = self.predictions[set_id]['residuals_test']
            X_test = self.data[set_id]['X_test']
            feat_names = self.model_feature_names[set_id]
            top_idx = importance_dict[set_id][:top_features]
            for j, feat_idx in enumerate(top_idx):
                ax = axes[i, j]
                feat_vals = X_test[:, feat_idx] if hasattr(X_test, 'shape') else X_test.iloc[:, feat_idx]
                ax.scatter(feat_vals, residuals, alpha=0.4, s=15,
                           color=self.model_colors[set_id], edgecolors='white')
                ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
                ax.set_xlabel(feat_names[feat_idx])
                if j == 0:
                    ax.set_ylabel("Residuals")
                ax.set_title(f"Model {set_id} – {feat_names[feat_idx]}")
                ax.grid(True, alpha=0.1)
                self._remove_spines(ax)
        fig.suptitle("Residuals vs Top Important Features", fontweight='bold', y=1.02)
        self.save_plot(fig, "16_error_vs_feature")

    def generate_all_comparisons(self):
        """Run all available comparison plots."""
        print("=" * 70)
        print("📊 Generating Multi‑Model Neural Network Comparison Plots (absolute raw values)")
        print("=" * 70)
        self.plot_parallel_coordinates()
        self.plot_metrics_bar_chart()
        self.plot_radar_chart()
        self.plot_comparative_learning_curves()
        self.plot_residuals_with_loess()
        self.plot_qq_plots()
        self.plot_cross_validation_comparison()
        self.plot_bootstrap_boxplots()
        self.plot_prediction_intervals()
        if SHAP_AVAILABLE:
            self.plot_comparative_shap()
        self.plot_feature_importance_comparison()
        self.plot_actual_vs_predicted_all()
        # Additional NN‑specific plots
        self.plot_residual_density_comparison()
        self.plot_train_test_metric_comparison()
        if TF_AVAILABLE:
            self.plot_weight_histograms()
        self.plot_error_vs_feature()
        print("\n✅ All NN comparative plots saved in:", self.results_path)




# Calling Zone

In [7]:
# Data initiation 
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# =============================================================================================================================================================================================
def data_caller(ModelSet):
    #Data fixation for the model
    DoNormalize = True
    Z_ScoreScaler = True                       #If false then MinMax Scaler is used in the modelling
    ModelSet = ModelSet                        #1 denotes fixed base (SF all upto 4), 2 denotes SSI only case (SF upto 1 and inputheads include Layers) 3 denotes total datasets, (SF upto 1 , excluding soil layers ie Fixed similar)
    includeCategoricalData = True               #True sets the categorical datta intact and False triggers the one hot encoding to the categorical data
    X, y = dataPreparation(DoNormalize = DoNormalize,Z_ScoreScaler = Z_ScoreScaler, ModelSet = ModelSet ,includeCategoricalData = includeCategoricalData  )
    yscaled = y * 100
    
    X.shape
    # X.columns
    
    # # Execute display of data statistics
    # # =============================================================================================================================================================================================
    
    # # # Generate comprehensive statistics
    # # generate_complete_data_statistics(X, y)
    
    # # # Generate LaTeX table
    # # generate_latex_table(X, y)
    return X, yscaled 

# Callers Section 

In [8]:
import joblib
import os
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
def SingleNNPlotter(ModelSet):

    # ResultsPath must be defined globally or passed as argument
    # Here we assume it is defined in the outer scope
    # If not, define it: ResultsPath = r"C:\Users\Acer\Documents\Machine Learning\CODES FINAL\NN\Results"

    X_clean, y_clean = data_caller(ModelSet)
    package_path = os.path.join(ResultsPath, f"NeuralNetwork_Set_{ModelSet}.joblib")
    package = joblib.load(package_path)

    # Get the model filename from the package
    model_filename = package["model_path"]  # e.g., "NeuralNetwork_Set_3.keras"
    
    # Build full path: if model_filename is not absolute, join with ResultsPath
    if not os.path.isabs(model_filename):
        model_full_path = os.path.join(ResultsPath, model_filename)
    else:
        model_full_path = model_filename

    # Load the Keras model
    model = load_model(model_full_path, custom_objects={'mae': 'mae'})

    feature_names = package["feature_names"]
    history_dict = package.get("history", None)

    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )

    visualizer = NeuralNetworkModernVisualizer(
        model=model,
        X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test,
        feature_names=feature_names,
        history=history_dict,
        results_path="./nn_modern_plots"
    )

    print("📊 Model Performance Metrics (Test):")
    print(visualizer.metrics['test'])
    visualizer.generate_all_plots()



# Assuming ResultsPath and data_caller are defined in your environment
# ResultsPath = "path/to/results"
# from your_data_module import data_caller

def DataExtractionFrom_NNModernVisualizer_touseinMultiplePlots():
    """
    Extract all necessary data from Neural Network models for ModelSet = 1,2,3.
    Returns:
        model_paths: dict {ModelSet: path_to_joblib}
        datasets:    dict {ModelSet: {'X_train', 'X_test', 'y_train', 'y_test', 'feature_names'}}
        visualizer_data: dict {ModelSet: {metrics, predictions, residuals, feature_names, model, etc.}}
    """
    model_paths = {}
    datasets = {}
    visualizer_data = {}

    for ModelSet in range(1, 4):
        # 1. Load the saved NN model package
        package_path = os.path.join(ResultsPath, f"NeuralNetwork_Set_{ModelSet}.joblib")
        if not os.path.exists(package_path):
            # Fallback to current directory
            package_path = f"NeuralNetwork_Set_{ModelSet}.joblib"
        package = joblib.load(package_path)
        
        
        # Get the model filename from the package
        model_filename = package["model_path"]  # e.g., "NeuralNetwork_Set_3.keras"
        
        # Build full path: if model_filename is not absolute, join with ResultsPath
        if not os.path.isabs(model_filename):
            model_full_path = os.path.join(ResultsPath, model_filename)
        else:
            model_full_path = model_filename
    
        # Load the Keras model
        model = load_model(model_full_path, custom_objects={'mae': 'mae'})


        # model = package["model"]
        feature_names = package["feature_names"]

        # 2. Prepare data (same split as during training)
        X_clean, y_clean = data_caller(ModelSet)
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_clean, test_size=0.2, random_state=42
        )

        # 3. Create a temporary visualizer instance to compute predictions and residuals
        temp_path = os.path.join(ResultsPath, f"temp_nn_viz_{ModelSet}")
        viz = NeuralNetworkModernVisualizer(   # Replace with your actual NN visualizer class name
            model=model,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            feature_names=feature_names,
            results_path=temp_path
        )

        # 4. Extract everything we might need
        nn_specific = {}
        # Some NN frameworks (e.g., Keras/TensorFlow) store history or weights differently
        if hasattr(model, 'history'):
            nn_specific['history'] = model.history
        if hasattr(model, 'get_weights'):
            nn_specific['weights'] = model.get_weights()
        if hasattr(model, 'get_config'):
            nn_specific['config'] = model.get_config()

        visualizer_data[ModelSet] = {
            # Core metrics (e.g., mse, mae, r2, etc. – ensure your visualizer provides these)
            'metrics': viz.metrics,
            # Predictions and residuals
            'y_pred_train': viz.y_pred_train,
            'y_pred_test': viz.y_pred_test,
            'residuals_train': viz.residuals_train,
            'residuals_test': viz.residuals_test,
            # Data splits (already aligned)
            'X_train': viz.X_train,
            'X_test': viz.X_test,
            'y_train': viz.y_train,
            'y_test': viz.y_test,
            # Feature names
            'feature_names': viz.feature_names,
            # The model itself
            'model': model,
            # NN specific additional attributes (adapt to your framework)
            **nn_specific,
        }

        # 5. Also store the plain datasets for convenience
        datasets[ModelSet] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test,
            'feature_names': X_clean.columns.tolist() if hasattr(X_clean, 'columns') else feature_names
        }
        model_paths[ModelSet] = package_path

        # Optionally clean up temporary plot directory
        # import shutil
        # shutil.rmtree(temp_path, ignore_errors=True)

    return model_paths, datasets, visualizer_data

# Example usage:
# _, _, nn_viz_data = DataExtractionFrom_NNModernVisualizer_touseinMultiplePlots()

# Simple and Main call

In [11]:
ResultsPath = r"C:\Users\Acer\Documents\Machine Learning\CODES FINAL\NN\Results"

ModelSet = 3
SingleNNPlotter(ModelSet)


# # Example usage (after extracting data with the NN extraction function):
_, _, nn_viz_data = DataExtractionFrom_NNModernVisualizer_touseinMultiplePlots()
comparator = MultiModelNNComparator(nn_viz_data, results_path="./nn_comparison_results")
comparator.generate_all_comparisons()

📊 Model Performance Metrics (Test):
{'R2': 0.9339634371723112, 'RMSE': np.float64(0.1283305560689576), 'MAE': 0.07750306888305918, 'MAPE': 0.1679068993802378, 'Explained Variance': 0.9341338260506704}
🎨 Neural Network Modern Visualization Suite

📊 EDA Plots:
  ✓ Saved: 01_correlation_heatmap.png
  ✓ Saved: 02_distributions.png
  ✓ Saved: 03_boxplots.png
  ✓ Saved: 04_feature_scatter.png
  ✓ Saved: 05_missing_values.png
  ✓ Saved: 06_outlier_detection.png

📈 Performance & Residuals:
  ✓ Saved: 07_actual_vs_predicted.png
  ✓ Saved: 08_residuals_vs_predicted.png
  ✓ Saved: 09_residual_distribution.png
  Learning curves skipped (NN may need more custom implementation): list index out of range
  ✓ Saved: 11_cumulative_error.png
  ✓ Saved: 20_performance_metrics.png
  ✓ Saved: 21_prediction_distributions.png

🔍 Feature Importance:
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
  ✓ Saved: 12_top_features.png
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
  ✓ Saved: 13_cumulative_features.png
  Permutation 

  ✓ Saved: 26_model_dashboard.png

✅ Generated 24 plots in: ./nn_modern_plots



📊 Neural Network Model Performance Summary (Test Set) – Absolute Values
       R²    RMSE     MAE    MAPE Expl. Var
1  0.9390  0.1384  0.0747  0.1552    0.9411
2  0.9494  0.1137  0.0773  0.1719    0.9498
3  0.9340  0.1283  0.0775  0.1679    0.9341

📊 Generating Multi‑Model Neural Network Comparison Plots (absolute raw values)
  ✓ Saved: 01_parallel_coordinates.png
  ✓ Saved: 02_metrics_bar_chart.png
  ✓ Saved: 03_radar_chart.png
Learning curve for Model 1 failed: list index out of range
Learning curve for Model 2 failed: list index out of range
Learning curve for Model 3 failed: list index out of range
  ✓ Saved: 04_comparative_learning_curves.png
  ✓ Saved: 05_residuals_loess.png
  ✓ Saved: 06_qq_plots.png
CV failed for Model 1: Cannot clone object '<Sequential name=sequential_21, built=True>' (type <class 'keras.src.models.sequential.Sequential'>): it does not seem to be a scikit-learn estimator as it does not implement a 'get_params' method.
CV failed for Model 2: Cannot clone obje